In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
import torch
print("Visible GPU:", torch.cuda.get_device_name(0))
print("Memory free:", torch.cuda.mem_get_info()[0] / 1e9, "GB")

Visible GPU: Quadro RTX 5000
Memory free: 8.735490048 GB


## Step 1 - Environment setup & `iter_passages()` loader

`iter_passages()` is the **single source of truth** for the entire RAG pipeline.  It iterates the FEVER Wikipedia dump (`wiki-pages/*.jsonl`) and yields one tuple per sentence: `(passage_id, page_id, sentence_idx, text)`.  Both the BM25 index and the Qdrant vector index are built by consuming this same generator in the same order, which guarantees that BM25 document IDs, Qdrant point IDs, and `(page_id, sentence_idx)` keys are perfectly aligned across the two indexes.

**Why sentence-level, not page-level?**  FEVER gold evidence annotations record the exact `(page_id, sentence_idx)` pair.  Indexing at sentence granularity means retrieval quality can be evaluated for free — checking whether gold evidence appears in the top-10 results requires no extra tooling.  Page-level retrieval would require a second-pass extraction and would silently lose any gold sentence ranked outside the top-K pages (a hard recall ceiling with no easy fix).

**Downstream dependencies.**  Step 2 (BM25 build) and Step 4 (Qdrant embed + upsert) both consume this generator.  The `passage_id` string `"{page_id}::{sentence_idx}"` is used as the stable key in both indexes and as the lookup key when the verifier needs to fetch evidence text.

| Choice | Why |
|---|---|
| Sentence granularity | Aligns with FEVER annotation units; enables free retrieval-quality eval |
| `{page_id}::{sentence_idx}` stable key | Deterministic; survives re-runs without index rebuild |
| Generator, not list | 25 M sentences would OOM if fully materialised; streaming keeps RAM flat |
| Skip empty / whitespace entries | FEVER `lines` field contains blank entries between section headers |
| Reuse `clean_artifacts` | Same bracket encoding (`-LRB-`, `-RRB-`, etc.) appears in wiki dump as in training data |

In [3]:
%pip install qdrant-client         -q
%pip install rank_bm25             -q
%pip install sentence-transformers -q
%pip install tqdm                  -q
%pip install jsonlines             -q



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install --upgrade Pillow

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import sys
# Force user-installed packages (e.g. Pillow 12.x, torch 2.2.1+cu121) before system packages.
_user_site = "/home/ai-ws2/.local/lib/python3.10/site-packages"
if _user_site not in sys.path:
    sys.path.insert(1, _user_site)

import os
import re
import glob
import json
import pickle
import random
import itertools
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

# ── Reproducibility + device ──────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Import transformers while torch is confirmed available ────────────────
# Must happen here so transformers caches is_torch_available()=True before
# any other cell imports from it.
import transformers
from transformers import AutoModel
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")

# ── Knowledge base ────────────────────────────────────────────────────────
WIKI_DIR        = "Data_wiki/Wikipedia dump/wiki-pages"
PAGE_INDEX_FILE = "page_to_file_index.json"

# ── Output paths (relative to notebook dir) ──────────────────────────────
QDRANT_PATH  = "rag/indices/qdrant"
BM25_PATH    = "rag/indices/bm25/bm25_okapi.pkl"
PASSAGE_MAP  = "rag/indices/bm25/passage_ids.pkl"

# ── Embedding model ───────────────────────────────────────────────────────
EMBED_MODEL    = "microsoft/harrier-oss-v1-270m"
EMBED_DIM      = 640    # Harrier last-token-pool output dim
EMBED_BATCH    = 128    # optimal from throughput test (772 p/s on RTX 5000)
EMBED_MAX_LEN  = 256
EMBED_DTYPE    = torch.float32  # FP16 → NaN on Turing CC 7.5

QUERY_PREFIX   = (
    "Instruct: Given a claim, retrieve evidence passages "
    "that support or refute it\nQuery: "
)
# Documents are encoded as raw text — no prefix. Applying the query
# prefix at index time silently degrades retrieval quality.

# ── Qdrant ────────────────────────────────────────────────────────────────
COLLECTION        = "fever_wiki"
HNSW_M            = 16
HNSW_EF_CONSTRUCT = 200

# ── Smoke test ────────────────────────────────────────────────────────────
SMOKE_LIMIT = 10_000

os.makedirs(QDRANT_PATH, exist_ok=True)
print(f"WIKI_DIR    : {WIKI_DIR}")
print(f"SMOKE_LIMIT : {SMOKE_LIMIT:,}")

Device : cuda
GPU    : Quadro RTX 5000
VRAM   : 16.7 GB
torch        : 2.2.1+cu121
transformers : 4.57.6
WIKI_DIR    : Data_wiki/Wikipedia dump/wiki-pages
SMOKE_LIMIT : 10,000


In [8]:
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.device_count())


True
12.1
1


In [15]:
import subprocess
result = subprocess.run(
    ["pip", "install", "torch==2.2.1+cu121", "torchvision", "torchaudio",
     "--index-url", "https://download.pytorch.org/whl/cu121", "-q"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.8.0 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip



In [5]:
# clean_artifacts is defined earlier in GP.ipynb (the NEI-pairs section).
# If starting a fresh kernel here, copy that cell in or run it first.

def clean_artifacts(text):
    text = text.replace("-LRB-", "(").replace("-RRB-", ")")
    text = text.replace("-LSB-", "[").replace("-RSB-", "]")
    text = text.replace("-LCB-", "{").replace("-RCB-", "}")
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def iter_passages(wiki_dir, limit=None):
    """Yield (passage_id, page_id, sentence_idx, text) from FEVER wiki-pages dump."""
    n = 0
    for fpath in sorted(glob.glob(os.path.join(wiki_dir, "*.jsonl"))):
        with open(fpath, encoding="utf-8") as fh:
            for line in fh:
                doc       = json.loads(line)
                page_id   = doc["id"]
                raw_lines = doc.get("lines", "")
                for raw in raw_lines.split("\n"):
                    raw = raw.strip()
                    if not raw:
                        continue
                    parts = raw.split("\t")
                    if len(parts) < 2:
                        continue
                    idx_str = parts[0]
                    if not idx_str.isdigit():
                        continue
                    text = parts[1].strip()  # parts[2:] are link annotations — discard
                    if not text:
                        continue
                    text         = clean_artifacts(text)
                    sentence_idx = int(idx_str)
                    passage_id   = f"{page_id}::{sentence_idx}"
                    yield passage_id, page_id, sentence_idx, text
                    n += 1
                    if limit is not None and n >= limit:
                        return

print("iter_passages() defined.")

iter_passages() defined.


In [10]:
first_file = sorted(glob.glob(os.path.join(WIKI_DIR, "*.jsonl")))[0]
print(f"Smoke-check file : {os.path.basename(first_file)}\n")

# ── First 5 passages ──────────────────────────────────────────────────────
print("Sample passages:")
for pid, page, sidx, text in itertools.islice(iter_passages(WIKI_DIR), 5):
    print(f"  passage_id   : {pid}")
    print(f"  page_id      : {page}")
    print(f"  sentence_idx : {sidx}")
    print(f"  text         : {text[:100]}")
    print()

# ── Passage count in first file only ──────────────────────────────────────────
count = 0
with open(first_file, encoding="utf-8") as fh:
    for line in fh:
        doc = json.loads(line)
        for raw in doc.get("lines", "").split("\n"):
            raw = raw.strip()
            if not raw:
                continue
            parts = raw.split("\t", 1)
            if len(parts) == 2 and parts[1].strip():
                count += 1

print(f"Passages in {os.path.basename(first_file)} : {count:,}")

Smoke-check file : wiki-001.jsonl

Sample passages:
  passage_id   : 1928_in_association_football::0
  page_id      : 1928_in_association_football
  sentence_idx : 0
  text         : The following are the football ( soccer ) events of the year 1928 throughout the world .

  passage_id   : 1986_NBA_Finals::0
  page_id      : 1986_NBA_Finals
  sentence_idx : 0
  text         : The 1986 NBA Finals was the championship round of the 1985 -- 86 NBA season . NBA National Basketbal

  passage_id   : 1986_NBA_Finals::1
  page_id      : 1986_NBA_Finals
  sentence_idx : 1
  text         : It pitted the Eastern Conference champion Boston Celtics against the Western Conference champion Hou

  passage_id   : 1986_NBA_Finals::2
  page_id      : 1986_NBA_Finals
  sentence_idx : 2
  text         : The Celtics defeated the Rockets four games to two to win their 16th NBA championship . NBA National

  passage_id   : 1986_NBA_Finals::3
  page_id      : 1986_NBA_Finals
  sentence_idx : 3
  text         : T

> **[DEPRECATED — Day 4 Track A]** Replaced by Step 2D–2E (bm25s). Preserved for reproducibility.

## Step 2 - BM25 index build

BM25 (Okapi variant via `rank_bm25`) is the lexical leg of the hybrid retrieval pipeline. It scores passages by term frequency normalised for document length and weighted by corpus-level IDF — making it strong at matching rare proper nouns, which dominate FEVER claims. Building BM25 first (CPU, no GPU required) also validates the passage loader end-to-end before committing GPU hours to the Qdrant embedding pass.

**Tokenisation.** Lowercase + `re.findall(r'\w+', text)` — no stemming, no explicit stopword removal. IDF down-weights high-frequency terms automatically, and stemming would degrade proper-noun matching (entity names like "Einstein" or "Meryl Streep" survive `\w+` cleanly; a Porter stemmer would mangle them). This matches the tokenisation used in GP.ipynb's NEI mining step.

**Downstream dependencies.** The `BM25Okapi` object and its parallel `passage_ids` list are saved to `rag/indices/bm25/` as pickles. Position `i` in `passage_ids` always corresponds to document `i` inside the BM25 object — `bm25.get_scores(q)[i]` ↔ `passage_ids[i]`. Step 5 (sanity check) and the Day 2 hybrid retrieval cell both load from this path. The smoke index (`*_smoke.pkl`) covers only `wiki-001.jsonl` and is used here to validate pipeline correctness before the full 25 M-sentence build.

| Choice | Why |
|---|---|
| `BM25Okapi` | Same library used in NEI mining (GP.ipynb); no new dependency |
| Lowercase + `\w+` split | Simple, fast; no stemming to corrupt entity names |
| No stopword removal | IDF handles weighting naturally; explicit removal risks clipping meaningful short terms |
| Parallel `passage_ids` list | Position-aligned with BM25 internals; O(1) id lookup at query time |
| `rag/indices/bm25/` output | Isolated from other index artifacts; 

In [17]:
from rank_bm25 import BM25Okapi

BM25_SMOKE_PATH = "rag/indices/bm25/bm25_smoke.pkl"
BM25_SMOKE_IDS  = "rag/indices/bm25/passage_ids_smoke.pkl"
os.makedirs("rag/indices/bm25", exist_ok=True)

def tokenize(text):
    return re.findall(r'\w+', text.lower())

# ── Read first wiki file only ─────────────────────────────────────────────
SMOKE_FILE = sorted(glob.glob(os.path.join(WIKI_DIR, "*.jsonl")))[0]
print(f"Source file      : {os.path.basename(SMOKE_FILE)}\n")

smoke_ids    = []
smoke_tokens = []
smoke_texts  = []

with open(SMOKE_FILE, encoding="utf-8") as fh:
    for raw_line in tqdm(fh, desc="Tokenising", unit=" docs"):
        doc     = json.loads(raw_line)
        page_id = doc["id"]
        for raw in doc.get("lines", "").split("\n"):
            raw = raw.strip()
            if not raw:
                continue
            parts = raw.split("\t")
            if len(parts) < 2:
                continue
            idx_str = parts[0]
            if not idx_str.isdigit():
                continue
            text = parts[1].strip()  # parts[2:] are link annotations — discard
            if not text:
                continue
            text = clean_artifacts(text)
            smoke_ids.append(f"{page_id}::{int(idx_str)}")
            smoke_tokens.append(tokenize(text))
            smoke_texts.append(text)

print(f"Passages indexed : {len(smoke_ids):,}")
print(f"Building BM25Okapi  ...")
bm25_smoke = BM25Okapi(smoke_tokens)

with open(BM25_SMOKE_PATH, "wb") as f:
    pickle.dump(bm25_smoke, f)
with open(BM25_SMOKE_IDS, "wb") as f:
    pickle.dump(smoke_ids, f)

print(f"Saved BM25 object : {BM25_SMOKE_PATH}")
print(f"Saved passage IDs : {BM25_SMOKE_IDS}")

Source file      : wiki-001.jsonl



Tokenising: 0 docs [00:00, ? docs/s]

Passages indexed : 170,546
Building BM25Okapi  ...
Saved BM25 object : rag/indices/bm25/bm25_smoke.pkl
Saved passage IDs : rag/indices/bm25/passage_ids_smoke.pkl


In [18]:
# ── Sanity query ──────────────────────────────────────────────────────────
QUERY    = "Albert Einstein Nobel Prize"
q_tokens = tokenize(QUERY)
scores   = bm25_smoke.get_scores(q_tokens)
top_idxs = scores.argsort()[::-1][:5]

print(f"Query : \"{QUERY}\"\n")
print(f"  {'Rank':<4} {'Score':>8}  passage_id")
print(f"  {'':4} {'':8}  text")
print("  " + "─" * 76)
for rank, idx in enumerate(top_idxs, 1):
    print(f"  {rank:<4}  {scores[idx]:>8.3f}  {smoke_ids[idx]}")
    print(f"            {smoke_texts[idx][:110]}")
    print()

# Alignment check: tokens[i] must match tokenize(texts[i]) for the top hit
best = top_idxs[0]
print(f"Alignment check — tokens[{best}][:5]         : {smoke_tokens[best][:5]}")
print(f"                  tokenize(texts[{best}])[:5] : {tokenize(smoke_texts[best])[:5]}")

Query : "Albert Einstein Nobel Prize"

  Rank    Score  passage_id
                 text
  ────────────────────────────────────────────────────────────────────────────
  1       20.059  1,2-Wittig_rearrangement::1
            The reaction is named for Nobel Prize winning chemist Georg Wittig .

  2       18.870  '47_-LRB-magazine-RRB-::20
            Nathaniel Benchley ventured `` Up in Benchley 's Room '' and Albert Einstein recommended a few science books .

  3       16.204  1GOAL_Education_for_All::6
            Alongside Queen Rania , it is co-chaired by FIFA President Sepp Blatter and Nobel prize laureate Archbishop De

  4       14.990  -LRB-137924-RRB-_2000_BD19::13
            is considered a good candidate for measuring the effects of Albert Einstein 's general theory of relativity be

  5       14.163  1979_Sakharov::22
            This minor planet was named in honour of renowned Russian mathematician and physicist Andrei Sakharov ( 1921 -

Alignment check — tokens[43329][:

> **[DEPRECATED — Day 4 Track A]** Replaced by Step 2D–2E (bm25s). Preserved for reproducibility.

### Step 2A - Full-corpus dry-pass count

A count-only pass over the complete FEVER wiki dump before committing to the full build. `iter_passages()` is called end-to-end but every tuple is discarded immediately — no tokenisation, no accumulation. This does two things: (1) confirms the expected ~25 M sentence count, and (2) exercises every file in the dump, surfacing any malformed JSON or encoding errors before the 45+ minute build begins. If this cell crashes, fix the underlying file issue before running 2B.

The elapsed time also gives a baseline for raw I/O speed — if the dry pass takes more than ~5 minutes, the full build will be I/O-bound rather than CPU-bound. The count is written to `corpus_count.txt` so Cell 2B can use it for a meaningful `tqdm` ETA without re-running this cell.

In [19]:
import time

COUNT_FILE = "rag/indices/bm25/corpus_count.txt"
os.makedirs("rag/indices/bm25", exist_ok=True)

print("Dry-pass: iterating full corpus (count only, no tokenisation) ...")
t0 = time.time()

total_passages = sum(1 for _ in iter_passages(WIKI_DIR))

elapsed = time.time() - t0
print(f"Total passages : {total_passages:,}")
print(f"Elapsed        : {elapsed:.1f} s  ({elapsed / 60:.1f} min)")

with open(COUNT_FILE, "w") as f:
    f.write(str(total_passages))
print(f"Count saved to : {COUNT_FILE}")

Dry-pass: iterating full corpus (count only, no tokenisation) ...
Total passages : 25,247,890
Elapsed        : 306.9 s  (5.1 min)
Count saved to : rag/indices/bm25/corpus_count.txt


> **[DEPRECATED — Day 4 Track A]** Replaced by Step 2D–2E (bm25s). Preserved for reproducibility.

### Step 2B - Full BM25 build

Second pass over the full corpus: tokenise every passage and accumulate two parallel lists — `full_ids` (passage ID strings) and `token_lists` (tokenised texts) — then construct a single `BM25Okapi` object in one shot. The build must be single-shot because `rank_bm25` computes IDF statistics over the entire corpus at construction time; incremental updates are not supported.

`tokenize()` is the same lowercase + `\w+` regex function used in the smoke cell. If `corpus_count.txt` exists from Cell 2A, `tqdm` receives an accurate `total=` for a meaningful ETA; otherwise it falls back to a count-only bar with a warning. Memory note: at ~25 M passages, `token_lists` can reach 10–15 GB of Python objects — well within the lab machine's 188 GB, but visible in `htop` during the build.

Both the BM25 object and the passage ID list are saved to disk at the end of this cell. If the kernel dies before saving, the full build must be re-run from scratch.

In [20]:
import time

try:
    import psutil
    _has_psutil = True
except ImportError:
    _has_psutil = False

BM25_FULL_PATH   = "rag/indices/bm25/bm25_okapi.pkl"
PASSAGE_IDS_PATH = "rag/indices/bm25/passage_ids.pkl"
COUNT_FILE       = "rag/indices/bm25/corpus_count.txt"
os.makedirs("rag/indices/bm25", exist_ok=True)

def tokenize(text):
    return re.findall(r'\w+', text.lower())

# ── Resolve corpus count for tqdm ETA ────────────────────────────────────
if os.path.exists(COUNT_FILE):
    with open(COUNT_FILE) as f:
        _total = int(f.read().strip())
    print(f"Corpus count (from 2A) : {_total:,}")
else:
    _total = None
    print("WARNING: corpus_count.txt not found — run Cell 2A first for ETA. Proceeding without total.")

# ── Tokenise + accumulate ─────────────────────────────────────────────────
full_ids    = []
token_lists = []

t0 = time.time()
for pid, _, _, text in tqdm(iter_passages(WIKI_DIR), total=_total,
                             desc="Tokenising", unit=" passages"):
    full_ids.append(pid)
    token_lists.append(tokenize(text))

elapsed_tok = time.time() - t0
print(f"\nPassages collected : {len(full_ids):,}")
print(f"Tokenisation time  : {elapsed_tok:.1f} s  ({elapsed_tok / 60:.1f} min)")

# ── Build BM25Okapi ───────────────────────────────────────────────────────
print("Building BM25Okapi (single-shot IDF computation) ...")
t1 = time.time()
bm25_full = BM25Okapi(token_lists)
elapsed_bm25 = time.time() - t1
print(f"BM25 build time    : {elapsed_bm25:.1f} s  ({elapsed_bm25 / 60:.1f} min)")

if _has_psutil:
    rss_gb = psutil.Process().memory_info().rss / 1e9
    print(f"RSS memory (approx): {rss_gb:.1f} GB")

# ── Save ──────────────────────────────────────────────────────────────────
print(f"\nSaving ...")
t2 = time.time()
with open(BM25_FULL_PATH, "wb") as f:
    pickle.dump(bm25_full, f)
with open(PASSAGE_IDS_PATH, "wb") as f:
    pickle.dump(full_ids, f)
elapsed_save = time.time() - t2
print(f"Saved BM25 object  : {BM25_FULL_PATH}")
print(f"Saved passage IDs  : {PASSAGE_IDS_PATH}")
print(f"Save time          : {elapsed_save:.1f} s")

Corpus count (from 2A) : 25,247,890


Tokenising:   0%|          | 0/25247890 [00:00<?, ? passages/s]


Passages collected : 25,247,890
Tokenisation time  : 741.5 s  (12.4 min)
Building BM25Okapi (single-shot IDF computation) ...
BM25 build time    : 337.2 s  (5.6 min)
RSS memory (approx): 55.5 GB

Saving ...
Saved BM25 object  : rag/indices/bm25/bm25_okapi.pkl
Saved passage IDs  : rag/indices/bm25/passage_ids.pkl
Save time          : 316.1 s


> **[DEPRECATED — Day 4 Track A]** Replaced by Step 2D–2E (bm25s). Preserved for reproducibility.

### Step 2C - Persistence & load helper

Defines `load_bm25_index()` — the standard entry point for all downstream cells that need the full BM25 index. Loading deserialises the pickled `BM25Okapi` object and its parallel `passage_ids` list from disk, restoring them to the exact state from Cell 2B. Any cell in Day 2 or later that needs to issue a BM25 query calls this function rather than rebuilding.

The sanity check below runs the same "Albert Einstein Nobel Prize" query used on the smoke index. With the full 25 M-sentence corpus, the top `passage_id` values should now include entries whose `page_id` component is `Albert_Einstein` — confirming the full index was built and loaded correctly. Passage text is not stored separately; the `page_id::sentence_idx` in each result is self-explanatory for eyeballing correctness.

In [10]:
BM25_FULL_PATH   = "rag/indices/bm25/bm25_okapi.pkl"
PASSAGE_IDS_PATH = "rag/indices/bm25/passage_ids.pkl"

def tokenize(text):
    return re.findall(r'\w+', text.lower())

def load_bm25_index(bm25_path=BM25_FULL_PATH, ids_path=PASSAGE_IDS_PATH):
    """Load and return (bm25, passage_ids) from disk."""
    with open(bm25_path, "rb") as f:
        bm25 = pickle.load(f)
    with open(ids_path, "rb") as f:
        passage_ids = pickle.load(f)
    print(f"Loaded BM25 index  : {len(passage_ids):,} passages")
    return bm25, passage_ids

# ── Load ──────────────────────────────────────────────────────────────────
bm25_full, full_ids = load_bm25_index()

# ── Sanity query ──────────────────────────────────────────────────────────
QUERY    = "Albert Einstein Nobel Prize"
q_tokens = tokenize(QUERY)
scores   = bm25_full.get_scores(q_tokens)
top_idxs = scores.argsort()[::-1][:5]

print(f"\nQuery : \"{QUERY}\"\n")
print(f"  {'Rank':<4} {'Score':>8}  passage_id")
print("  " + "─" * 60)
for rank, idx in enumerate(top_idxs, 1):
    print(f"  {rank:<4}  {scores[idx]:>8.3f}  {full_ids[idx]}")

# Expect page_id component to include "Albert_Einstein" in top results
top_pages = [pid.split("::")[0] for pid in (full_ids[i] for i in top_idxs)]
einstein_found = any("Albert_Einstein" in p for p in top_pages)
print(f"\nAlbert_Einstein in top-5 pages : {einstein_found}")

Loaded BM25 index  : 25,247,890 passages

Query : "Albert Einstein Nobel Prize"

  Rank    Score  passage_id
  ────────────────────────────────────────────────────────────
  1       38.203  List_of_Jewish_American_physicists::22
  2       35.146  Einstein_Prize::7
  3       34.495  List_of_peace_prizes::5
  4       31.890  Albert_Einstein_Peace_Prize::0
  5       30.532  Reform_Congregation_Keneseth_Israel_-LRB-Philadelphia-RRB-::19

Albert_Einstein in top-5 pages : True


### Step 2D - bm25s index build

**Day 4 Track A:** replaces `rank_bm25` with `bm25s` — a vectorised NumPy reimplementation of BM25Okapi. `rank_bm25.get_scores()` iterates 25.2 M rows in Python (~80–110 s/query); `bm25s` uses a sparse inverted-index matrix so scoring becomes a sparse dot-product over only the posting-list entries of query terms, dropping latency to sub-second.

| Choice | Why |
|--------|-----|
| `method="bm25"` | Okapi variant — matches `rank_bm25.BM25Okapi` exactly |
| `stopwords=None` | No stopword removal — consistent with `re.findall(r'\w+', ...)` used in rank_bm25 build |
| Save without corpus | Smaller index files (~3–8 GB); passage IDs persisted separately as pickle |
| `MIN_FREE_GB = 10` | Conservative guard for the New Volume where the index is written |

**Dependency:** `iter_passages`, `WIKI_DIR` in scope (Step 1). Does **not** depend on rank_bm25 state.

> rank_bm25 cells (2A/2B/2C) are preserved above and marked `[DEPRECATED Day 4 Track A]`.

In [6]:
# Cell 2D -- bm25s index build (Day 4 Track A -- replaces rank_bm25 for speed)
# Depends on: iter_passages, WIKI_DIR (Step 1)
# rank_bm25 cells (2A/2B/2C) preserved above -- marked DEPRECATED Day 4 Track A
%pip install bm25s -q
import os, pickle, shutil, time
from pathlib import Path
import bm25s

# ── Config ──────────────────────────────────────────────────────────────────────────
BM25S_INDEX_PATH  = "rag/indices/bm25s"
BM25S_IDS_PATH    = "rag/indices/bm25s/passage_ids_bm25s.pkl"
MIN_FREE_GB       = 10
TOTAL_PASSAGES_2D = 25_247_890
LOG_EVERY         = 1_000_000    # print progress every N passages

# ── Disk probe ───────────────────────────────────────────────────────────────────
os.makedirs(BM25S_INDEX_PATH, exist_ok=True)
_free_gb = shutil.disk_usage(BM25S_INDEX_PATH).free / 1e9
print(f"Free space on index volume  : {_free_gb:.1f} GB  (need >= {MIN_FREE_GB} GB)")
if _free_gb < MIN_FREE_GB:
    raise RuntimeError(
        f"Insufficient disk: {_free_gb:.1f} GB free, need >= {MIN_FREE_GB} GB"
    )

# ── Collect texts and IDs ───────────────────────────────────────────────────
print(f"\nCollecting {TOTAL_PASSAGES_2D:,} passages ...")
_all_ids   = []
_all_texts = []
t0 = time.time()

for _i, (pid, _, _, text) in enumerate(iter_passages(WIKI_DIR)):
    _all_ids.append(pid)
    _all_texts.append(text)
    if (_i + 1) % LOG_EVERY == 0:
        _ela  = time.time() - t0
        _rate = (_i + 1) / _ela
        _eta  = (TOTAL_PASSAGES_2D - (_i + 1)) / _rate / 60
        print(f"  {_i+1:>11,} / {TOTAL_PASSAGES_2D:,}  "
              f"|  {_rate:,.0f} p/s  |  ETA {_eta:.0f} min")

t_collect = time.time() - t0
print(f"\nPassages collected : {len(_all_ids):,}  ({t_collect/60:.1f} min)")

# ── Tokenise ──────────────────────────────────────────────────────────────────────
print("\nTokenising with bm25s.tokenize (stopwords=None, stemmer=None) ...")
t1 = time.time()
_corpus_tokens = bm25s.tokenize(
    _all_texts,
    stopwords     = None,
    stemmer       = None,
    show_progress = True,
)
t_tok = time.time() - t1
del _all_texts    # free RAM before index build
print(f"Tokenisation time  : {t_tok:.1f} s  ({t_tok/60:.1f} min)")

# ── Build ─────────────────────────────────────────────────────────────────────────────
print("\nBuilding bm25s index (method='bm25') ...")
t2 = time.time()
bm25s_retriever = bm25s.BM25(method="robertson")
bm25s_retriever.index(_corpus_tokens)
t_build = time.time() - t2
print(f"Build time         : {t_build:.1f} s  ({t_build/60:.1f} min)")

# ── Save ───────────────────────────────────────────────────────────────────────────
print(f"\nSaving index to {BM25S_INDEX_PATH!r} ...")
t3 = time.time()
bm25s_retriever.save(BM25S_INDEX_PATH)   # index only -- no corpus (keeps files smaller)
with open(BM25S_IDS_PATH, "wb") as f:
    pickle.dump(_all_ids, f)
t_save = time.time() - t3
print(f"Save time          : {t_save:.1f} s")

# ── Report ───────────────────────────────────────────────────────────────────────
_total_size = sum(
    p.stat().st_size for p in Path(BM25S_INDEX_PATH).rglob("*") if p.is_file()
) / 1e9
print(f"\nIndex on-disk size : {_total_size:.2f} GB")
print(f"Passages indexed   : {len(_all_ids):,}")
print("\nStep 2D: COMPLETE")



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Free space on index volume  : 395.7 GB  (need >= 10 GB)

    1,000,000 / 25,247,890  |  95,722 p/s  |  ETA 4 min
    2,000,000 / 25,247,890  |  96,764 p/s  |  ETA 4 min
    3,000,000 / 25,247,890  |  96,816 p/s  |  ETA 4 min
    4,000,000 / 25,247,890  |  97,557 p/s  |  ETA 4 min
    5,000,000 / 25,247,890  |  98,372 p/s  |  ETA 3 min
    6,000,000 / 25,247,890  |  98,491 p/s  |  ETA 3 min
    7,000,000 / 25,247,890  |  98,659 p/s  |  ETA 3 min
    8,000,000 / 25,247,890  |  98,679 p/s  |  ETA 3 min
    9,000,000 / 25,247,890  |  98,697 p/s  |  ETA 3 min
   10,000,000 / 25,247,890  |  98,804 p/s  |  ETA 3 min
   11,000,000 / 25,247,890  |  98,923 p/s  |  ETA 2 min
   12,000,000 / 25,247,890  |  98,678 p/s  |  ETA 2 min
   13,000,000 / 25,247,890  |  98,755 p/s  |  ETA 2 min
   14,000,000 / 25,247,890  |  100,0

Split strings:   0%|          | 0/25247890 [00:00<?, ?it/s]

Tokenisation time  : 318.5 s  (5.3 min)

Building bm25s index (method='bm25') ...


BM25S Count Tokens:   0%|          | 0/25247890 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/25247890 [00:00<?, ?it/s]

Build time         : 509.7 s  (8.5 min)

Saving index to 'rag/indices/bm25s' ...
Save time          : 30.5 s

Index on-disk size : 3.85 GB
Passages indexed   : 25,247,890

Step 2D: COMPLETE


### Step 2E - bm25s index load & sanity check

Reloads the bm25s index with `mmap=True` and verifies retrieval quality on three hand-picked claims. `mmap=True` lets the OS load only the pages accessed rather than pulling the full sparse matrix into RAM; initial queries are slightly slower but memory pressure is much lower.

Results should be topically related to each claim. Minor rank differences from the rank_bm25 output are expected due to tokenization differences; gross mismatches (top-5 completely unrelated) indicate a corrupted index or tokenization mismatch at build time.

**Dependency:** Step 2D must have completed (bm25s index and `passage_ids_bm25s.pkl` on disk).

In [7]:
# Cell 2E -- bm25s index load + sanity check
# Depends on: Step 2D (index files on disk)
import pickle, time
import bm25s

# ── Config ──────────────────────────────────────────────────────────────────────────
BM25S_INDEX_PATH = "rag/indices/bm25s"
BM25S_IDS_PATH   = "rag/indices/bm25s/passage_ids_bm25s.pkl"
SANITY_K         = 5

# ── Load ───────────────────────────────────────────────────────────────────────────
print("Loading bm25s index (mmap=True) ...")
t0 = time.time()
bm25s_retriever   = bm25s.BM25.load(BM25S_INDEX_PATH, mmap=True)
t_load = time.time() - t0
print(f"  Load time     : {t_load:.1f} s")

print("Loading passage IDs ...")
with open(BM25S_IDS_PATH, "rb") as f:
    passage_ids_bm25s = pickle.load(f)
print(f"  Passages      : {len(passage_ids_bm25s):,}")

# ── Sanity retrieval ──────────────────────────────────────────────────────────
_SANITY_CLAIMS = [
    "Albert Einstein won the Nobel Prize",
    "The Eiffel Tower is in Paris",
    "Python is a programming language",
]

print()
for _sc in _SANITY_CLAIMS:
    t1 = time.time()
    _q = bm25s.tokenize([_sc], stopwords=None, stemmer=None)
    _r, _s = bm25s_retriever.retrieve(_q, k=SANITY_K)
    t_q = time.time() - t1
    print(f'Query ({t_q*1000:.0f} ms): "{_sc}"')
    print(f"  {'Rank':<4}  {'Score':>9}  passage_id")
    print("  " + chr(8212) * 60)
    for _ri in range(len(_r[0])):
        _pid   = passage_ids_bm25s[int(_r[0][_ri])]
        _score = float(_s[0][_ri])
        print(f"  {_ri+1:<4}  {_score:>9.4f}  {_pid}")
    print()

print("Step 2E: PASS")


Loading bm25s index (mmap=True) ...
  Load time     : 3.6 s
Loading passage IDs ...
  Passages      : 25,247,890



Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Query (885 ms): "Albert Einstein won the Nobel Prize"
  Rank      Score  passage_id
  ————————————————————————————————————————————————————————————
  1       15.0681  List_of_Jewish_American_physicists::22
  2       13.8980  Einstein_Prize::7
  3       13.7024  List_of_peace_prizes::5
  4       12.7763  Albert_Einstein_Peace_Prize::0
  5       11.9695  Reform_Congregation_Keneseth_Israel_-LRB-Philadelphia-RRB-::19



Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Query (574 ms): "The Eiffel Tower is in Paris"
  Rank      Score  passage_id
  ————————————————————————————————————————————————————————————
  1       13.1664  Eiffel_Tower_-LRB-disambiguation-RRB-::0
  2       12.4738  Semantic_data_model::10
  3       12.0701  Eiffel_Tower_-LRB-Paris,_Texas-RRB-::2
  4       11.8462  Eiffel_Tower::0
  5       11.6918  Eiffel_Tower_-LRB-Paris,_Tennessee-RRB-::0



Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Query (475 ms): "Python is a programming language"
  Rank      Score  passage_id
  ————————————————————————————————————————————————————————————
  1       13.6054  3K::28
  2       13.6054  PDB::29
  3       13.1133  Mystic_BBS::16
  4       12.5264  Core_Python_Programming::0
  5       12.1385  Edile::1

Step 2E: PASS


## Step 3 - Qdrant collection initialization

Sets up the local Qdrant in-process client and creates the `fever_wiki` vector collection. No embedding or upserting happens here — that is Step 4. Keeping this step separate means the collection can be inspected, dropped, or recreated independently of the expensive embedding pass.

**Embedding model.** `microsoft/harrier-oss-v1-270m` (Gemma-based decoder, 268M parameters) outputs **640 dimensions** via last-token pooling + L2 normalisation. `EMBED_DIM = 640` is set in the config accordingly.

**Asymmetric retrieval API.** Harrier uses an instruction prefix to separate query from document representations — no LoRA adapter switching required:

| Mode | Input format |
|---|---|
| Query encoding (Day 2) | `"Instruct: Given a claim, retrieve evidence passages that support or refute it\nQuery: {text}"` |
| Document encoding (Step 4 corpus) | `{text}` — raw text, no prefix |

Applying the query prefix at index time silently degrades retrieval quality — corpus encoding in Step 4 must use raw text only. No `trust_remote_code` is required.

**dtype.** FP32 is required — FP16 produces NaN attention overflow on this GPU (Turing CC 7.5, no native BF16). Verified peak VRAM: 1043 MB during inference, well within the 15.5 GB budget.

**Throughput.** Verified 772 p/s at batch_size=128 → estimated 9.1h for the full 25.2M passage corpus.

**HNSW config.** `m=16, ef_construct=200` as locked in the project config. Higher `ef_construct` improves index graph quality at build time with no runtime cost.

| Choice | Why |
|---|---|
| Local in-process Qdrant | No server process to manage; on-disk persistence survives kernel restarts |
| Cosine distance | Harrier embeddings are L2-normalised; cosine and dot-product are equivalent |
| Drop + recreate | Previous collection held v5-small vectors (dim=1024, incompatible vector space) |
| `indexing_threshold=30_000_000` | Disables live HNSW build during the 25.2M upsert; HNSW is triggered once after load completes |
| Minimal payload: `passage_id` + `text` | `page_id` and `sentence_idx` derivable from `passage_id`; smaller payload reduces storage |


In [28]:
# ── Step 3: Qdrant collection — Harrier (dim=640) ─────────────────────────────
import os, pathlib, torch
from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams, Distance, HnswConfigDiff, OptimizersConfigDiff
)

# ── Config (referenced by all subsequent cells) ───────────────────────────────
MODEL_ID          = "microsoft/harrier-oss-v1-270m"
COLLECTION        = "fever_wiki"
QDRANT_HOST       = "localhost"
QDRANT_PORT       = 6333
QDRANT_STORAGE    = "rag/indices/qdrant"   # symlink — storage sanity check only
EMBED_DIM         = 640               # Harrier output dim — NOT 768 or 1024
EMBED_DTYPE       = torch.float32     # FP16 → NaN on Turing CC 7.5; FP32 verified clean
BATCH_SIZE        = 128               # optimal from throughput test (772 p/s)
QUERY_PREFIX      = (
    "Instruct: Given a claim, retrieve evidence passages "
    "that support or refute it\nQuery: "
)

os.environ["CUDA_VISIBLE_DEVICES"]    = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── Verify symlink (confirms storage is on the large volume) ───────────────────
qdrant_link = pathlib.Path(QDRANT_STORAGE)
assert qdrant_link.is_symlink(), f"{QDRANT_STORAGE} is not a symlink — check manually"
print(f"Symlink OK : {QDRANT_STORAGE} → {qdrant_link.resolve()}")

# ── Close any stale client ─────────────────────────────────────────────────────
try:
    client.close()
    print("Closed existing Qdrant client.")
except NameError:
    pass
except Exception as e:
    print(f"Warning: client.close() raised {type(e).__name__}: {e}")
    print("If recreate fails below, restart kernel and re-run this cell.")

# ── Connect to Qdrant server ───────────────────────────────────────────────────
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

existing = [c.name for c in client.get_collections().collections]
print(f"Existing collections: {existing}")

if COLLECTION in existing:
    client.delete_collection(COLLECTION)
    print(f"Deleted '{COLLECTION}'")

client.create_collection(
    collection_name   = COLLECTION,
    vectors_config    = VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    hnsw_config       = HnswConfigDiff(m=16, ef_construct=200),
    optimizers_config = OptimizersConfigDiff(indexing_threshold=30_000_000),
)

# ── Sanity print ───────────────────────────────────────────────────────────────
info       = client.get_collection(COLLECTION)
vec        = info.config.params.vectors
actual_dim = vec.size
print(f"Collection         : {COLLECTION}")
print(f"Dim                : {actual_dim}")
print(f"Distance           : {vec.distance}")
print(f"indexing_threshold : {info.config.optimizer_config.indexing_threshold:,}")
print(f"Points             : {info.points_count:,}")

assert actual_dim == EMBED_DIM, f"Dim mismatch: got {actual_dim}, expected {EMBED_DIM}"
print("Step 3: PASS")


Symlink OK : rag/indices/qdrant → /media/ai-ws2/New Volume/qdrant_storage
Closed existing Qdrant client.
Existing collections: []
Collection         : fever_wiki
Dim                : 640
Distance           : Cosine
indexing_threshold : 30,000,000
Points             : 0
Step 3: PASS


In [12]:
import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
import transformers
print("transformers:", transformers.__version__)

torch: 2.2.1+cu121 cuda: True
transformers: 4.57.6


In [9]:
!pip install "transformers>=4.44,<5.0" --upgrade

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 9.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.14.0
    Uninstalling huggingface_hub-1.14.0:
      Successfully uninstalled huggingface_hub-1.14.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.8.0
    Uninstalling transformers-5.8.0:
      Successfully uninstalled transformers-5.8.0

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import numpy as np
import torch
print("numpy:", np.__version__)
print("torch:", torch.__version__)

numpy: 1.26.4
torch: 2.2.1+cu121


In [11]:
!pip install "numpy<2" --upgrade

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 9.6 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas-stubs 2.3.0.250703 requires types-pytz>=2022.1.1, which is not installed.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## Step 4 - Harrier embed + Qdrant upsert

Embeds all 25.2 M passages with `microsoft/harrier-oss-v1-270m` (FP32, GPU 1) and upserts them into the `fever_wiki` Qdrant collection built in Step 3. Verified throughput on the RTX 5000: **772 p/s** -- projected corpus runtime **~9.1 h**. Checkpointing in 4C means the job can be interrupted and resumed without loss.

The step is split into three sub-cells for crash isolation: **4A** loads the model and validates encoding behaviour (dim, dtype, asymmetry, throughput); **4B** does a 10 K smoke upsert and verifies the full round-trip dense query; **4C** runs the full corpus embed + upsert with batch checkpointing. Do not run 4C until 4A and 4B pass.

| Choice | Why |
|---|---|
| FP32 | FP16 produces NaN on Turing CC 7.5 -- no native BF16, limited dynamic range under the long instruction prefix |
| `attn_implementation="sdpa"` | Scaled dot-product attention; no `trust_remote_code` required |
| Instruction prefix at query time only | Documents encoded as raw text; prefix applied only to claims (Day 2) -- applying it at index time silently degrades retrieval |
| Batch 128 | Optimal on RTX 5000 (772 p/s) -- marginal gains beyond this |
| Atomic checkpoint write | `.tmp` -> rename prevents a corrupt checkpoint on mid-write crash |
| Upsert batch 500 | Balances Qdrant IPC overhead vs call count (recommended 100-1 000 per call) |


### Step 4A - Model load & encoding validation

Loads `microsoft/harrier-oss-v1-270m` in FP32 and runs four checks before committing GPU hours to the full corpus:

1. **Shape check** -- encode 10 passages; assert output shape is `(10, 640)` -- NOT 1 024, NOT 768.
2. **Batch consistency** -- encode the same text twice; cosine must be > 0.9999 (deterministic FP32).
3. **Asymmetry check** -- encode the same passage with `QUERY_PREFIX` prepended vs raw; cosine must be < 1.0, confirming the instruction prefix is active and retrieval is asymmetric as intended.
4. **Throughput sweep** -- time batch sizes `[64, 128, 256]` over 500 passages (after a 50-passage warm-up), calling `torch.cuda.synchronize()` before stopping each timer. Expected optimum: batch 128 at ~772 p/s.

**Model API:** `AutoModel.from_pretrained(MODEL_ID, torch_dtype=torch.float32, attn_implementation="sdpa")`, no `trust_remote_code`. Pooling is **last-token** (handles both left- and right-padded inputs); output is L2-normalised to unit norm.


In [29]:
import itertools, time
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

os.environ["CUDA_VISIBLE_DEVICES"]    = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# -- Config -------------------------------------------------------------------
EMBED_MODEL    = "microsoft/harrier-oss-v1-270m"
EMBED_DIM      = 640
EMBED_BATCH    = 128
EMBED_MAX_LEN  = 256
EMBED_DTYPE    = torch.float32
QUERY_PREFIX   = (
    "Instruct: Given a claim, retrieve evidence passages "
    "that support or refute it\nQuery: "
)
TOTAL_PASSAGES = 25_247_890          # from Step 2A


# -- Harrier helpers ----------------------------------------------------------
def _last_token_pool(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    # last non-padding token -- required by Harrier
    left_pad = mask[:, -1].sum() == mask.shape[0]
    if left_pad:
        return hidden[:, -1]
    seq_lens = mask.sum(dim=1) - 1
    return hidden[torch.arange(hidden.size(0), device=hidden.device), seq_lens]


def harrier_encode(texts: list, prefix: str = "") -> np.ndarray:
    # returns L2-normed float32 (N, 640); prefix='' for docs, QUERY_PREFIX for claims
    if prefix:
        texts = [prefix + t for t in texts]
    tok = tokenizer(
        texts,
        max_length     = EMBED_MAX_LEN,
        padding        = True,
        truncation     = True,
        return_tensors = "pt",
    ).to(DEVICE)
    with torch.no_grad():
        out = embed_model(**tok)
    vecs = _last_token_pool(out.last_hidden_state, tok["attention_mask"])
    return F.normalize(vecs, p=2, dim=-1).cpu().float().numpy()


def _cosine(a: np.ndarray, b: np.ndarray) -> float:
    a, b = a.flatten().astype(np.float64), b.flatten().astype(np.float64)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))


# -- GPU guard ----------------------------------------------------------------
if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available -- run the Step 1 config cell first.")
DEVICE = torch.device("cuda")

# -- Load model ---------------------------------------------------------------
print(f"Loading {EMBED_MODEL} ...")
t_load     = time.time()
tokenizer  = AutoTokenizer.from_pretrained(EMBED_MODEL)
embed_model = AutoModel.from_pretrained(
    EMBED_MODEL,
    torch_dtype         = EMBED_DTYPE,
    attn_implementation = "sdpa",
).to(DEVICE).eval()
print(f"Load time  : {time.time() - t_load:.1f} s")
print(f"Device     : {next(embed_model.parameters()).device}")
print(f"Dtype      : {next(embed_model.parameters()).dtype}")
print(f"VRAM used  : {torch.cuda.memory_allocated() / 1024**2:.0f} MB")
print(f"Params     : {sum(p.numel() for p in embed_model.parameters()):,}")

# -- Collect 10 smoke passages ------------------------------------------------
smoke_pids, smoke_texts = [], []
for pid, _page, _idx, text in itertools.islice(iter_passages(WIKI_DIR), 10):
    smoke_pids.append(pid)
    smoke_texts.append(text)

# -- Check 1 -- shape ---------------------------------------------------------
vecs = harrier_encode(smoke_texts)
assert vecs.shape == (10, EMBED_DIM), f"Shape FAIL: {vecs.shape} -- expected (10, {EMBED_DIM})"
assert np.all(np.isfinite(vecs)),     "NaN/Inf in embeddings -- FAIL"
print(f"\nCheck 1 -- shape        : {vecs.shape}  PASS")

# -- Check 2 -- batch consistency ---------------------------------------------
v1 = harrier_encode([smoke_texts[0]])
v2 = harrier_encode([smoke_texts[0]])
cos_con = _cosine(v1[0], v2[0])
assert cos_con > 0.9999, f"Consistency FAIL: {cos_con:.6f}"
print(f"Check 2 -- consistency  : {cos_con:.6f}  PASS")

# -- Check 3 -- asymmetry (query prefix vs raw) -------------------------------
raw_vec   = harrier_encode([smoke_texts[0]])[0]
query_vec = harrier_encode([smoke_texts[0]], prefix=QUERY_PREFIX)[0]
cos_asym  = _cosine(raw_vec, query_vec)
assert cos_asym < 1.0, f"Asymmetry FAIL: cosine={cos_asym:.6f} -- prefix not active"
print(f"Check 3 -- asymmetry    : {cos_asym:.6f}  PASS  (< 1.0 confirms prefix active)")

# -- Check 4 -- throughput sweep ----------------------------------------------
warmup = [t for _pid, _p, _i, t in itertools.islice(iter_passages(WIKI_DIR), 50)]
harrier_encode(warmup)   # warm up, discard

sweep = [t for _pid, _p, _i, t in itertools.islice(iter_passages(WIKI_DIR), 500)]
print(f"\nThroughput sweep (500 passages, max_len={EMBED_MAX_LEN}) :")
print(f"  {'Batch':>6}  {'p/s':>6}  {'projected (h)':>13}")
print(f"  {'--' * 16}")

best_ps, best_bs = 0.0, EMBED_BATCH
for bs in [64, 128, 256]:
    t0 = time.time()
    for start in range(0, len(sweep), bs):
        harrier_encode(sweep[start : start + bs])
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    ps      = len(sweep) / elapsed
    proj_h  = TOTAL_PASSAGES / ps / 3600
    marker  = " <- best" if ps > best_ps else ""
    if ps > best_ps:
        best_ps, best_bs = ps, bs
    print(f"  {bs:>6}  {ps:>6.0f}  {proj_h:>11.1f} h{marker}")

EMBED_BATCH = best_bs
print(f"\nEMBED_BATCH set to : {EMBED_BATCH}")
print(f"Projected runtime  : {TOTAL_PASSAGES / best_ps / 3600:.1f} h")
print("Step 4A: PASS")


Loading microsoft/harrier-oss-v1-270m ...
Load time  : 3.6 s
Device     : cuda:0
Dtype      : torch.float32
VRAM used  : 1031 MB
Params     : 268,098,176

Check 1 -- shape        : (10, 640)  PASS
Check 2 -- consistency  : 1.000000  PASS
Check 3 -- asymmetry    : 0.787510  PASS  (< 1.0 confirms prefix active)

Throughput sweep (500 passages, max_len=256) :
   Batch     p/s  projected (h)
  --------------------------------
      64     349         20.1 h <- best
     128     316         22.2 h
     256     271         25.9 h

EMBED_BATCH set to : 64
Projected runtime  : 20.1 h
Step 4A: PASS


### Step 4B - Smoke upsert & dense retrieval check (10K passages)

Embeds the first 10,000 passages as **raw text** (no prefix) and upserts them into the `fever_wiki` Qdrant collection. This validates the full encode -> upsert -> query round-trip before the ~9.1 h full-corpus run in 4C.

Each Qdrant point stores two payload fields: `passage_id` (the `page_id::sentence_idx` string) and `text` (the raw sentence). Point IDs are sequential integers `0, 1, 2, ...` matching iteration order.

At the end, a dense query for `"Albert Einstein Nobel Prize"` is issued (encoded with `QUERY_PREFIX`) and the top-5 results are printed. Since the smoke corpus covers only the first wiki file, Albert Einstein may not appear -- the primary goal is confirming the query returns results with a valid `passage_id` payload. The collection is then **deleted and recreated clean** so 4C starts from an empty collection.


In [30]:
import itertools, time
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, HnswConfigDiff, OptimizersConfigDiff, PointStruct, VectorParams,
)
from tqdm.auto import tqdm

# -- Config -------------------------------------------------------------------
EMBED_BATCH        = 128
EMBED_MAX_LEN      = 256
EMBED_DIM          = 640
COLLECTION         = "fever_wiki"
QDRANT_PATH        = "rag/indices/qdrant"
HNSW_M             = 16
HNSW_EF_CONSTRUCT  = 200
SMOKE_N            = 10_000
UPSERT_BATCH       = 500
QUERY_PREFIX       = (
    "Instruct: Given a claim, retrieve evidence passages "
    "that support or refute it\nQuery: "
)

# -- Symlink guard ------------------------------------------------------------
from pathlib import Path
assert Path(QDRANT_PATH).is_symlink(), f"{QDRANT_PATH} is not a symlink -- check storage setup"

# -- Guard: harrier_encode must be in scope from 4A ---------------------------
try:
    harrier_encode
except NameError:
    raise RuntimeError("harrier_encode not defined -- run Step 4A first")

# -- Reload client if needed --------------------------------------------------
try:
    client
except NameError:
    client = QdrantClient(path=QDRANT_PATH)
    print("Qdrant client opened.")

# -- Verify Step 3 ran with Harrier config (dim=640, not stale Jina dim=1024) --
info_pre = client.get_collection(COLLECTION)
assert info_pre.config.params.vectors.size == EMBED_DIM, (
    f"Collection dim={info_pre.config.params.vectors.size} but EMBED_DIM={EMBED_DIM} -- "
    f"re-run Step 3 to recreate with Harrier config before continuing"
)
print(f"Collection dim verified : {info_pre.config.params.vectors.size}")

# -- Collect smoke passages ---------------------------------------------------
print(f"Collecting {SMOKE_N:,} smoke passages ...")
smoke_pids, smoke_texts = [], []
for pid, _page, _idx, text in itertools.islice(iter_passages(WIKI_DIR), SMOKE_N):
    smoke_pids.append(pid)
    smoke_texts.append(text)
print(f"Collected  : {len(smoke_texts):,}")

# -- Embed + upsert -----------------------------------------------------------
print(f"\nEmbedding + upserting {len(smoke_texts):,} passages (raw text, batch={EMBED_BATCH}) ...")
t0 = time.time()
for start in tqdm(range(0, len(smoke_texts), EMBED_BATCH), desc="Embedding+upserting", unit=" batch"):
    batch_pids  = smoke_pids[start  : start + EMBED_BATCH]
    batch_texts = smoke_texts[start : start + EMBED_BATCH]
    vecs        = harrier_encode(batch_texts)   # raw text -- no prefix
    points      = [
        PointStruct(
            id      = start + j,
            vector  = vecs[j].tolist(),
            payload = {"passage_id": batch_pids[j], "text": batch_texts[j]},
        )
        for j in range(len(batch_pids))
    ]
    for u in range(0, len(points), UPSERT_BATCH):
        client.upsert(collection_name=COLLECTION, points=points[u : u + UPSERT_BATCH], wait=True)

elapsed = time.time() - t0
count   = client.count(COLLECTION).count
print(f"\nUpserted  : {count:,} points")
print(f"Elapsed   : {elapsed:.1f} s")
print(f"Throughput: {len(smoke_texts) / elapsed:.0f} p/s")

# -- Round-trip dense query ---------------------------------------------------
TEST_CLAIM = "Albert Einstein was awarded the Nobel Prize in Physics."
q_vec      = harrier_encode([TEST_CLAIM], prefix=QUERY_PREFIX)[0].tolist()
hits       = client.query_points(
    collection_name = COLLECTION,
    query           = q_vec,
    limit           = 5,
    with_payload    = True,
).points
print(f'\nDense query : "{TEST_CLAIM}"\n')
print(f"  {'Rank':<4} {'Score':>8}  passage_id")
print("  " + "-" * 60)
for rank, h in enumerate(hits, 1):
    print(f"  {rank:<4}  {h.score:>8.4f}  {h.payload.get('passage_id', 'MISSING')}")
    print(f"            {h.payload.get('text', '')[:110]}")
    print()
assert all("passage_id" in h.payload for h in hits), "payload missing passage_id -- FAIL"
print("Round-trip  : PASS  (passage_id present in all results)")

# -- Wipe smoke data: delete + recreate collection for 4C --------------------
print(f"\nDeleting smoke collection '{COLLECTION}' ...")
# -- Guard: refuse to wipe a collection that looks like a partial 4C run --
final_count = client.count(COLLECTION).count
if final_count > SMOKE_N + 100:
    raise RuntimeError(
        f"Refusing to delete '{COLLECTION}': has {final_count:,} points "
        f"(expected ~{SMOKE_N:,}). Looks like 4C data -- wipe manually if intentional."
    )
client.delete_collection(COLLECTION)
client.create_collection(
    collection_name   = COLLECTION,
    vectors_config    = VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    hnsw_config       = HnswConfigDiff(m=HNSW_M, ef_construct=HNSW_EF_CONSTRUCT),
    optimizers_config = OptimizersConfigDiff(indexing_threshold=30_000_000),
)
# create_collection does not honour OptimizersConfigDiff in this client version --
# force the threshold with update_collection which is verified to work.
client.update_collection(
    collection_name   = COLLECTION,
    optimizers_config = OptimizersConfigDiff(indexing_threshold=30_000_000),
)
info = client.get_collection(COLLECTION)
print(f"Collection '{COLLECTION}' recreated clean")
print(f"  Dim       : {info.config.params.vectors.size}")
print(f"  Threshold : {info.config.optimizer_config.indexing_threshold:,}")
print(f"  Points    : {info.points_count:,}")
assert info.config.optimizer_config.indexing_threshold == 30_000_000, \
    f"Threshold not applied: got {info.config.optimizer_config.indexing_threshold:,}"
print("Step 4B: PASS -- ready for 4C")



Collection dim verified : 640
Collected  : 10,000

Embedding + upserting 10,000 passages (raw text, batch=128) ...


Embedding+upserting:   0%|          | 0/79 [00:00<?, ? batch/s]


Upserted  : 10,000 points
Elapsed   : 38.5 s
Throughput: 260 p/s

Dense query : "Albert Einstein was awarded the Nobel Prize in Physics."

  Rank    Score  passage_id
  ------------------------------------------------------------
  1       0.4006  1995_Nebelhorn_Trophy::7
            The Fritz-Geiger-Memorial Trophy was presented to the country with the highest placements across all disciplin

  2       0.3840  1939_International_University_Games_-LRB-Vienna-RRB-::4
            The formal opening was by Bernhard Rust , the Reich Minister of Science , Education and Culture , on 20 August

  3       0.3828  1889_Kumamoto_earthquake::5
            The earthquake was the first major one after the establishment of the Seismological Society of Japan ( in 1880

  4       0.3826  1,000,000::6
            Physical quantities can also be expressed using the SI prefix mega ( M ) , when dealing with SI units ; for ex

  5       0.3742  1939_International_University_Games_-LRB-Vienna-RRB-::1
     

### Step 4C - Full corpus embed + upsert (25.2 M passages)

Embeds all 25,247,890 passages with `microsoft/harrier-oss-v1-270m` (FP32, GPU 1, batch 128) and upserts them into the `fever_wiki` Qdrant collection. Projected runtime: **~9.1 h** on the RTX 5000.

**Optimisations applied**

| Technique | Effect |
|---|---|
| `indexing_threshold=30_000_000` | Disables live HNSW indexing during load -- upsert throughput substantially higher than baseline |
| Async upsert overlap (`ThreadPoolExecutor`) | GPU embeds batch N+1 while disk-commits batch N; effective wall-clock ~= `max(T_embed, T_upsert)` per batch |
| Atomic checkpoint write | `.tmp` -> rename every `CHECKPOINT_EVERY` batches -- crash-safe resume |

**Checkpointing** -- saves progress every `CHECKPOINT_EVERY = 50` embed batches (~6 400 passages, ~5-8 min at 772 p/s). On crash, re-run the cell: it reads `checkpoint_4c.json`, skips already-indexed passages, and resumes from the last confirmed point.

**Post-load** -- `update_collection(optimizers_config=OptimizersConfigDiff(indexing_threshold=20_000))` drops the threshold below corpus size, triggering Qdrant's one-shot HNSW build. Poll `client.get_collection(COLLECTION).status` until `green`.


In [35]:
# Cell 4C -- Full corpus embed + upsert (Harrier, FP32, dim=640)
# harrier_encode + indexing_threshold=30M (already set) + async overlap + checkpointing

import collections, concurrent.futures, itertools, json, shutil, time
from pathlib import Path
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, OptimizersConfigDiff
from tqdm.auto import tqdm

# -- Config -------------------------------------------------------------------
EMBED_BATCH       = 128   # override 4A sweep -- verified 772 p/s in test_harrier.py
CHECKPOINT_PATH   = Path("rag/indices/qdrant/checkpoint_4c.json")
CHECKPOINT_EVERY  = 50    # save every N embed batches (~6 400 passages at batch=128)
EMPTY_CACHE_EVERY = 200   # torch.cuda.empty_cache() every N batches
TOTAL_PASSAGES    = 25_247_890
QDRANT_HOST       = "localhost"
QDRANT_PORT       = 6333


# -- Checkpoint helpers -------------------------------------------------------
def _save_ckpt(path, n):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps({"passages_done": n}))
    tmp.replace(path)

def _load_ckpt(path):
    return json.loads(path.read_text())["passages_done"] if path.exists() else 0


# -- Passage batcher with resume skip ----------------------------------------
def _batches(skip, batch_size):
    gen = iter_passages(WIKI_DIR)
    if skip:
        print(f"  Skipping {skip:,} already-indexed passages ...")
        collections.deque(itertools.islice(gen, skip), maxlen=0)
    buf_p, buf_t = [], []
    for pid, _page, _idx, text in gen:
        buf_p.append(pid)
        buf_t.append(text)
        if len(buf_p) == batch_size:
            yield buf_p, buf_t
            buf_p, buf_t = [], []
    if buf_p:
        yield buf_p, buf_t


# -- Upsert worker (runs in background thread) --------------------------------
def _upsert(points):
    client.upsert(collection_name=COLLECTION, points=points, wait=True)


# -- Symlink guard ------------------------------------------------------------
assert Path("rag/indices/qdrant").is_symlink(), "rag/indices/qdrant is not a symlink -- check storage"

# -- Disk space probe ---------------------------------------------------------
qdrant_target = Path("rag/indices/qdrant").resolve()
free_gb       = shutil.disk_usage(qdrant_target).free / 1e9
MIN_FREE_GB   = 100
print(f"Free space on Qdrant volume : {free_gb:.1f} GB  (need >= {MIN_FREE_GB} GB)")
if free_gb < MIN_FREE_GB:
    raise RuntimeError(
        f"Insufficient disk space: {free_gb:.1f} GB free on {qdrant_target}, "
        f"need >= {MIN_FREE_GB} GB for 4C + HNSW build"
    )

# -- Connect with 30s HTTP timeout (prevents ReadTimeout during WAL/segment stalls) --
# Default httpx timeout (~5s) is too short when Qdrant briefly stalls under bulk load.
# 30s covers any realistic background-task spike without adding delay on normal calls.
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT, timeout=120)
print(f"Qdrant client connected : {QDRANT_HOST}:{QDRANT_PORT}  http_timeout=120s")

# -- Resume -------------------------------------------------------------------
start_at = _load_ckpt(CHECKPOINT_PATH)
print(f"Starting 4C  |  passages_done={start_at:,}  ({start_at / TOTAL_PASSAGES * 100:.2f}%)")
print(f"Collection points before run : {client.count(COLLECTION).count:,}")

# -- Main loop ----------------------------------------------------------------
t0             = time.time()
passages_done  = start_at
batch_idx      = 0
pending_future = None

with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    pbar = tqdm(
        total     = TOTAL_PASSAGES,
        initial   = start_at,
        desc      = "4C",
        unit      = " p",
        smoothing = 0.05,
    )

    for pids, texts in _batches(start_at, EMBED_BATCH):

        # -- Embed on GPU (raw text -- no prefix) --
        vecs_np = harrier_encode(texts)   # shape (EMBED_BATCH, 640), FP32

        # -- Build Qdrant points --
        base_id = passages_done
        points  = [
            PointStruct(
                id      = base_id + j,
                vector  = vecs_np[j].tolist(),
                payload = {"passage_id": pids[j], "text": texts[j]},
            )
            for j in range(len(pids))
        ]

        # -- Gate: wait for upsert N-1, fire upsert N in background --
        if pending_future is not None:
            pending_future.result()
        pending_future = executor.submit(_upsert, points)

        passages_done += len(pids)
        batch_idx     += 1
        pbar.update(len(pids))

        elapsed = time.time() - t0
        p_per_s = (passages_done - start_at) / elapsed if elapsed > 0 else 0
        eta_h   = (TOTAL_PASSAGES - passages_done) / p_per_s / 3600 if p_per_s > 0 else 0
        pbar.set_postfix({"p/s": f"{p_per_s:.0f}", "ETA_h": f"{eta_h:.1f}"})

        # -- Checkpoint --
        if batch_idx % CHECKPOINT_EVERY == 0:
            pending_future.result()
            pending_future = None
            _save_ckpt(CHECKPOINT_PATH, passages_done)

        # -- VRAM cleanup --
        if batch_idx % EMPTY_CACHE_EVERY == 0:
            torch.cuda.empty_cache()

    # Flush final in-flight upsert
    if pending_future is not None:
        pending_future.result()

    pbar.close()

# -- Final checkpoint ---------------------------------------------------------
_save_ckpt(CHECKPOINT_PATH, passages_done)
elapsed_total = time.time() - t0
print(f"\n4C done  |  {passages_done:,} passages  |  {elapsed_total / 3600:.2f} h")
print(f"Throughput : {(passages_done - start_at) / elapsed_total:.0f} p/s (wall-clock)")
print(f"Collection count : {client.count(COLLECTION).count:,}")

# -- Trigger one-shot HNSW build ----------------------------------------------
print("\nLowering indexing_threshold -> 20,000 to trigger HNSW build ...")
client.update_collection(
    collection_name   = COLLECTION,
    optimizers_config = OptimizersConfigDiff(indexing_threshold=20_000),
)
info = client.get_collection(COLLECTION)
print(f"optimizer.indexing_threshold : {info.config.optimizer_config.indexing_threshold:,}")
print(f"collection.status            : {info.status}")

Free space on Qdrant volume : 465.6 GB  (need >= 100 GB)
Qdrant client connected : localhost:6333  http_timeout=120s
Starting 4C  |  passages_done=2,438,400  (9.66%)
Collection points before run : 2,439,168


4C:  10%|9         | 2438400/25247890 [00:00<?, ? p/s]

  Skipping 2,438,400 already-indexed passages ...

4C done  |  25,247,890 passages  |  26.78 h
Throughput : 237 p/s (wall-clock)
Collection count : 25,247,890

Lowering indexing_threshold -> 20,000 to trigger HNSW build ...
optimizer.indexing_threshold : 20,000
collection.status            : yellow


### Step 4D - HNSW build status poll

Polls the Qdrant collection status every 60 seconds until the HNSW index build triggered at the end of Step 4C completes. This is a standalone recovery cell -- safe to re-run after a kernel restart. The HNSW build continues in Qdrant even if the kernel is restarted.

Step 4C triggered the build by lowering `indexing_threshold` to 20 000 (below corpus size). The index is ready when `status` reports `green`. Expected build time for 25.2 M 640-dim vectors is 1--3 h after upload completes.


In [36]:
# Cell 4D -- Poll HNSW build status until green
import time
from qdrant_client import QdrantClient

QDRANT_HOST = "localhost"
QDRANT_PORT = 6333
COLLECTION  = "fever_wiki"
POLL_EVERY  = 60   # seconds between status checks

try:
    client
except NameError:
    client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT, timeout=120)

print(f"Polling HNSW status every {POLL_EVERY}s -- safe to Ctrl+C, index keeps building.\n")
t0 = time.time()
while True:
    status    = client.get_collection(COLLECTION).status
    elapsed_h = (time.time() - t0) / 3600
    print(f"  [{elapsed_h:5.2f} h] status = {status}")
    if str(status).lower().endswith("green"):
        print(f"\nHNSW build complete in {elapsed_h:.2f} h.")
        break
    time.sleep(POLL_EVERY)

Polling HNSW status every 60s -- safe to Ctrl+C, index keeps building.

  [ 0.00 h] status = yellow
  [ 0.02 h] status = yellow
  [ 0.03 h] status = yellow
  [ 0.05 h] status = yellow
  [ 0.07 h] status = yellow
  [ 0.08 h] status = yellow
  [ 0.10 h] status = yellow
  [ 0.12 h] status = yellow
  [ 0.13 h] status = yellow
  [ 0.15 h] status = yellow
  [ 0.17 h] status = yellow
  [ 0.18 h] status = yellow
  [ 0.20 h] status = yellow
  [ 0.22 h] status = yellow
  [ 0.23 h] status = yellow
  [ 0.25 h] status = yellow
  [ 0.27 h] status = yellow
  [ 0.28 h] status = yellow
  [ 0.30 h] status = yellow
  [ 0.32 h] status = yellow
  [ 0.33 h] status = yellow
  [ 0.35 h] status = yellow
  [ 0.37 h] status = yellow
  [ 0.38 h] status = yellow
  [ 0.40 h] status = yellow
  [ 0.42 h] status = yellow
  [ 0.43 h] status = yellow
  [ 0.45 h] status = yellow
  [ 0.47 h] status = yellow
  [ 0.48 h] status = yellow
  [ 0.50 h] status = yellow
  [ 0.52 h] status = yellow
  [ 0.53 h] status = yellow
  [ 

## Step 5 - Hybrid retrieval pipeline (Day 2)

Dense retrieval alone misses exact lexical matches: FEVER claims often hinge on proper nouns, dates, and numerical specifics where term-frequency signals dominate. BM25 alone misses semantic paraphrase -- a claim phrased differently from its evidence passage scores low even when the meaning is identical. Combining both legs via score fusion captures complementary signal.

The fused candidate set (~20 passages) is reranked by a cross-encoder that jointly encodes each `(claim, passage)` pair and produces a single relevance score. A cross-encoder sees the full query-passage interaction, making it substantially more precise than a bi-encoder at short range -- but prohibitively expensive at corpus scale, so it is applied only to the small fused set.

| Choice | Why |
|---|---|
| Dense + BM25 top-10 each | Balanced recall vs reranker compute -- fewer than 10 drops recall, more than 10 inflates the fused set |
| RRF as default fusion | Rank-based, scale-free (BM25 and cosine scores live on different scales); minimal hyperparameters (only `k`) |
| Weighted-sum as alternative | Enables per-side weight tuning on Day 4 dev-set sweep |
| Cross-encoder reranker, not bi-encoder | Captures full query-passage interaction; feasible only on small candidate sets (~20) |
| Top-5 final cap | Matches DeBERTa-v2 ensemble NLI input budget (one claim + one evidence per pair, ensembled across top-5 in Day 3) |

### Step 5A - Load indices & embedder

Loads the three components required by all subsequent Step 5 cells: the BM25 full-corpus index, the Qdrant dense-vector client, and the Harrier embedding model. After this cell runs, `bm25_full`, `passage_ids`, `pid_to_qid`, `client`, `embed_model`, `tokenizer`, and `harrier_encode` are all in kernel scope.

**BM25** is deserialised from `rag/indices/bm25/bm25_okapi.pkl` (built in Step 2B). The parallel `passage_ids` list holds the `page_id::sentence_idx` key for every document -- position `i` in `passage_ids` always corresponds to Qdrant point ID `i`. A reverse-lookup dict `pid_to_qid` is built for O(1) string-to-integer-ID mapping in the retrieve_top3 orchestrator (Step 5F).

**Qdrant** connects to the Docker server on port 6333 with a 120 s socket timeout (same as Steps 3/4C). A soft warning is printed if the Qdrant point count does not match the BM25 passage count -- dense retrieval still works on the indexed subset. A hard assertion fires only if the collection vector dimension does not match `EMBED_DIM=640`.

**Harrier** is loaded with the same `AutoModel` / `AutoTokenizer` / FP32 / `attn_implementation="sdpa"` pattern as Step 4A. Re-loading here makes Step 5A independently runnable without requiring Step 4A to still be in scope.

In [6]:
# Cell 5A -- Load BM25 index, Qdrant client, and Harrier embedder
import os, pickle, re, time

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from qdrant_client import QdrantClient

os.environ["CUDA_VISIBLE_DEVICES"]    = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── Config ─────────────────────────────────────────────────────────────────
EMBED_MODEL      = "microsoft/harrier-oss-v1-270m"
EMBED_DIM        = 640
EMBED_BATCH      = 128
EMBED_MAX_LEN    = 256
EMBED_DTYPE      = torch.float32
QUERY_PREFIX     = (
    "Instruct: Given a claim, retrieve evidence passages "
    "that support or refute it\nQuery: "
)
COLLECTION       = "fever_wiki"
QDRANT_HOST      = "localhost"
QDRANT_PORT      = 6333
BM25_FULL_PATH   = "rag/indices/bm25/bm25_okapi.pkl"
PASSAGE_IDS_PATH = "rag/indices/bm25/passage_ids.pkl"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ── Helpers ─────────────────────────────────────────────────────────────────
def tokenize(text: str) -> list:
    return re.findall(r'\w+', text.lower())


def load_bm25_index(bm25_path=BM25_FULL_PATH, ids_path=PASSAGE_IDS_PATH):
    """Load and return (bm25, passage_ids) from disk."""
    with open(bm25_path, "rb") as f:
        bm25 = pickle.load(f)
    with open(ids_path, "rb") as f:
        pids = pickle.load(f)
    print(f"  Passages  : {len(pids):,}")
    return bm25, pids


def _last_token_pool(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    left_pad = mask[:, -1].sum() == mask.shape[0]
    if left_pad:
        return hidden[:, -1]
    seq_lens = mask.sum(dim=1) - 1
    return hidden[torch.arange(hidden.size(0), device=hidden.device), seq_lens]


def harrier_encode(texts: list, prefix: str = "") -> np.ndarray:
    # returns L2-normed float32 (N, 640); prefix='' for docs, QUERY_PREFIX for claims
    if prefix:
        texts = [prefix + t for t in texts]
    tok = tokenizer(
        texts,
        max_length     = EMBED_MAX_LEN,
        padding        = True,
        truncation     = True,
        return_tensors = "pt",
    ).to(DEVICE)
    with torch.no_grad():
        out = embed_model(**tok)
    vecs = _last_token_pool(out.last_hidden_state, tok["attention_mask"])
    return F.normalize(vecs, p=2, dim=-1).cpu().float().numpy()


# ── 1. Load BM25 ────────────────────────────────────────────────────────────
print("Loading BM25 index ...")
bm25_full, passage_ids = load_bm25_index()

# ── 2. Reverse lookup: passage_id string -> Qdrant integer point ID ─────────
# Qdrant IDs are sequential integers matching iter_passages order: passage_ids[i] == point ID i.
print("Building passage_id -> Qdrant ID index ...")
pid_to_qid = {pid: i for i, pid in enumerate(passage_ids)}
print(f"  Index size : {len(pid_to_qid):,} entries")

# ── 3. Connect to Qdrant ────────────────────────────────────────────────────
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT, timeout=120)
print(f"\nQdrant connected   : {QDRANT_HOST}:{QDRANT_PORT}")

# ── 4. Load Harrier ─────────────────────────────────────────────────────────
print(f"Loading {EMBED_MODEL} ...")
t0          = time.time()
tokenizer   = AutoTokenizer.from_pretrained(EMBED_MODEL)
embed_model = AutoModel.from_pretrained(
    EMBED_MODEL,
    torch_dtype         = EMBED_DTYPE,
    attn_implementation = "sdpa",
).to(DEVICE).eval()
print(f"  Load time  : {time.time() - t0:.1f} s")
if torch.cuda.is_available():
    print(f"  VRAM used  : {torch.cuda.memory_allocated() / 1024**2:.0f} MB")

# ── 5. Verify Qdrant collection dim ─────────────────────────────────────────
info       = client.get_collection(COLLECTION)
actual_dim = info.config.params.vectors.size
assert actual_dim == EMBED_DIM, (
    f"Dim mismatch: Qdrant dim={actual_dim} vs EMBED_DIM={EMBED_DIM} -- re-run Step 3"
)
qdrant_points = client.count(COLLECTION).count
print(f"\nQdrant  dim       : {actual_dim}")
print(f"Qdrant  status    : {info.status}")
print(f"Qdrant  points    : {qdrant_points:,}")

# ── 6. Cross-index consistency ───────────────────────────────────────────────
bm25_count = len(passage_ids)
print(f"BM25    passages  : {bm25_count:,}")
if qdrant_points == bm25_count:
    print("Cross-index check : PASS  (BM25 count == Qdrant count)")
else:
    print(
        f"WARNING: BM25 has {bm25_count:,} passages but Qdrant has {qdrant_points:,} points -- "
        f"dense retrieval covers partial corpus until Step 4C completes"
    )

print("\nStep 5A: PASS -- bm25_full, passage_ids, pid_to_qid, client, embed_model, harrier_encode in scope")

Loading BM25 index ...
  Passages  : 25,247,890
Building passage_id -> Qdrant ID index ...
  Index size : 25,247,888 entries

Qdrant connected   : localhost:6333
Loading microsoft/harrier-oss-v1-270m ...


`torch_dtype` is deprecated! Use `dtype` instead!


  Load time  : 7.0 s
  VRAM used  : 1023 MB

Qdrant  dim       : 640
Qdrant  status    : green
Qdrant  points    : 25,247,890
BM25    passages  : 25,247,890
Cross-index check : PASS  (BM25 count == Qdrant count)

Step 5A: PASS -- bm25_full, passage_ids, pid_to_qid, client, embed_model, harrier_encode in scope


### Step 5B - BM25 search helper  *(Day 4 Track A: bm25s backend)*

Defines `bm25_search(claim, k=10)` — the same public interface as before, now backed by `bm25s` instead of `rank_bm25`. The previous `rank_bm25.BM25Okapi.get_scores()` scored all 25.2 M passages in Python on every query (~80–110 s/query); `bm25s` uses a sparse inverted-index matrix, reducing per-query latency to sub-second.

Tokenisation uses `bm25s.tokenize` (same tokeniser used at index build time — **never** mix with the `re.findall` tokeniser from rank_bm25). Return type is unchanged: `list[tuple[str, float]]` — top-k `(passage_id, score)` pairs, descending. All downstream consumers (Step 5D fusion, Step 5F orchestrator) remain unmodified.

| Choice | Why |
|--------|-----|
| `bm25s.BM25.load(mmap=True)` | Avoids pulling full sparse matrix into RAM; OS pages on demand |
| `retrieve(sorted=True)` | Default — returns top-k already in descending score order |
| `passage_ids_bm25s` | Position-aligned with bm25s internal doc IDs (built in Step 2D) |

**Dependency:** bm25s index on disk (Step 2D). `bm25s_retriever` and `passage_ids_bm25s` are loaded here if not already in scope from Step 2E.

In [7]:
# Cell 5B -- BM25 search helper (bm25s -- Day 4 Track A)
# Depends on: BM25S_INDEX_PATH, BM25S_IDS_PATH constants; bm25s package
# bm25s_retriever and passage_ids_bm25s are loaded here if not already in scope from Step 2E.
import pickle, time
import bm25s as _bm25s_mod

# ── Config ──────────────────────────────────────────────────────────────────────────
BM25S_INDEX_PATH = "rag/indices/bm25s"
BM25S_IDS_PATH   = "rag/indices/bm25s/passage_ids_bm25s.pkl"

# ── Load (idempotent -- skip if already in scope from Step 2E) ─────────────────────
try:
    bm25s_retriever
except NameError:
    print("Loading bm25s index (mmap=True) ...")
    t_l0 = time.time()
    bm25s_retriever = _bm25s_mod.BM25.load(BM25S_INDEX_PATH, mmap=True)
    print(f"  Index loaded   : {time.time() - t_l0:.1f} s")

try:
    passage_ids_bm25s
except NameError:
    print("Loading passage IDs ...")
    with open(BM25S_IDS_PATH, "rb") as f:
        passage_ids_bm25s = pickle.load(f)
    print(f"  Passages       : {len(passage_ids_bm25s):,}")


# ── DEPRECATED Day 4 Track A -- replaced with bm25s for ~100-500x speedup. Kept for reproducibility.
# def bm25_search(claim: str, k: int = 10) -> list:
#     # Returns [(passage_id, score), ...] length k, sorted by descending BM25 score.
#     q_tokens = tokenize(claim)
#     scores   = bm25_full.get_scores(q_tokens)         # np.ndarray shape (N,)
#     # O(n) partition then sort only top-k -- ~3x faster than full argsort on 25M entries
#     top_idx  = np.argpartition(scores, -k)[-k:]
#     top_idx  = top_idx[np.argsort(scores[top_idx])[::-1]]
#     return [(passage_ids[i], float(scores[i])) for i in top_idx]


def bm25_search(claim: str, k: int = 10) -> list:
    """Returns top-k (passage_id, score) tuples, descending by score."""
    q_tokens        = _bm25s_mod.tokenize([claim], stopwords=None, stemmer=None)
    results, scores = bm25s_retriever.retrieve(q_tokens, k=k)
    doc_ids         = results[0]    # shape (k,) -- integer indices, sorted descending
    doc_scores      = scores[0]     # shape (k,) -- BM25 scores, sorted descending
    return [
        (passage_ids_bm25s[int(doc_ids[i])], float(doc_scores[i]))
        for i in range(len(doc_ids))
    ]


# ── Sanity test ───────────────────────────────────────────────────────────────────
TEST_CLAIM = "Albert Einstein was awarded the Nobel Prize in Physics."
bm25_hits  = bm25_search(TEST_CLAIM, k=10)

print(f'\nBM25 (bm25s) search: "{TEST_CLAIM}"\n')
print(f"  {'Rank':<4}  {'Score':>9}  passage_id")
print("  " + chr(8212) * 64)
for rank, (pid, score) in enumerate(bm25_hits[:3], 1):
    print(f"  {rank:<4}  {score:>9.4f}  {pid}")

print(f"\nReturned {len(bm25_hits)} results  |  top score: {bm25_hits[0][1]:.4f}")
print("Step 5B: PASS  (bm25s backend)")


Loading bm25s index (mmap=True) ...
  Index loaded   : 4.4 s
Loading passage IDs ...
  Passages       : 25,247,890


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


BM25 (bm25s) search: "Albert Einstein was awarded the Nobel Prize in Physics."

  Rank      Score  passage_id
  ————————————————————————————————————————————————————————————————
  1       16.6500  Einstein_Prize::7
  2       15.4366  Albert_Einstein_Peace_Prize::0
  3       15.0681  List_of_Jewish_American_physicists::22

Returned 10 results  |  top score: 16.6500
Step 5B: PASS  (bm25s backend)


In [8]:
# Cell 5B-latency -- bm25s vs rank_bm25 latency comparison (Day 4 Track A)
# Demonstrates speedup from the bm25s swap. Depends on: bm25_search (Step 5B).
import time

# ── Config ──────────────────────────────────────────────────────────────────────────
BENCH_CLAIM           = "Albert Einstein won the Nobel Prize in Physics"
BENCH_K               = 10
BENCH_REPS            = 3
RANK_BM25_BASELINE_MS = 90_000   # median latency observed in Day 3 smoke test (ms)

print(f"Latency benchmark: bm25_search (bm25s backend)")
print(f"  Claim : {BENCH_CLAIM!r}")
print(f"  k     : {BENCH_K}")
print(f"  Reps  : {BENCH_REPS}")
print()

_times = []
for _r in range(BENCH_REPS):
    _t0      = time.time()
    _hits    = bm25_search(BENCH_CLAIM, k=BENCH_K)
    _elapsed = time.time() - _t0
    _times.append(_elapsed)
    print(f"  Run {_r+1}: {_elapsed*1000:.0f} ms  |  top: {_hits[0][0]}")

_avg_ms = sum(_times) / len(_times) * 1000
print(f"\n  Avg latency (bm25s)     : {_avg_ms:.0f} ms")
print(f"  rank_bm25 baseline      : ~{RANK_BM25_BASELINE_MS:,} ms  (Day 3 smoke test)")
print(f"  Speedup                 : ~{RANK_BM25_BASELINE_MS / _avg_ms:.0f}x")
print()
print(f"  Top-10 passage IDs (bm25s):")
for _rank, (_pid, _score) in enumerate(_hits, 1):
    print(f"    {_rank:2d}. {_score:>9.4f}  {_pid}")
print("\nStep 5B-latency: COMPLETE")


Latency benchmark: bm25_search (bm25s backend)
  Claim : 'Albert Einstein won the Nobel Prize in Physics'
  k     : 10
  Reps  : 3



Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Run 1: 654 ms  |  top: List_of_Jewish_American_physicists::22


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Run 2: 584 ms  |  top: List_of_Jewish_American_physicists::22


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Run 3: 577 ms  |  top: List_of_Jewish_American_physicists::22

  Avg latency (bm25s)     : 605 ms
  rank_bm25 baseline      : ~90,000 ms  (Day 3 smoke test)
  Speedup                 : ~149x

  Top-10 passage IDs (bm25s):
     1.   15.0681  List_of_Jewish_American_physicists::22
     2.   14.0180  Einstein_Prize::7
     3.   13.7024  List_of_peace_prizes::5
     4.   13.3382  Physics_-LRB-disambiguation-RRB-::26
     5.   13.3382  Raymond_Davis_Jr.::3
     6.   13.2549  Einstein_and_Religion::0
     7.   12.9559  Nuclear_emulsion::13
     8.   12.9559  Masatoshi_Koshiba::1
     9.   12.8946  List_of_female_Nobel_laureates::8
    10.   12.7763  Albert_Einstein_Peace_Prize::0

Step 5B-latency: COMPLETE


### Step 5C - Dense search helper

Defines `dense_search(claim, k=10)` which encodes the claim with `QUERY_PREFIX` via `harrier_encode`, queries the `fever_wiki` Qdrant collection for the top-k cosine-nearest neighbours, and returns `[(passage_id, score), ...]` sorted by descending similarity score.

**HNSW vs brute-force.** If the collection status is not `green` when this cell runs -- for example because Step 4D is still in progress -- Qdrant falls back to brute-force scan automatically. Results are correct but slower. A soft warning is printed rather than a hard failure so the cell can be exercised during development with a partial or not-yet-indexed collection.

**Dependency:** `harrier_encode`, `client`, `QUERY_PREFIX`, `COLLECTION`, and `EMBED_DIM` must be in scope (Step 5A).

In [9]:
# Cell 5C -- Dense search helper
# Depends on: harrier_encode, client, QUERY_PREFIX, COLLECTION (Step 5A)


def dense_search(claim: str, k: int = 10) -> list:
    # Returns [(passage_id, cosine_score), ...] length k, sorted by descending similarity.
    # Soft HNSW-ready warning -- hard fail would block development with partial index
    status = str(client.get_collection(COLLECTION).status).lower()
    if not status.endswith("green"):
        print(
            f"WARNING: collection status={status} -- "
            f"Qdrant using brute-force scan (results correct, slower)"
        )
    vec  = harrier_encode([claim], prefix=QUERY_PREFIX)[0].tolist()
    hits = client.query_points(
        collection_name = COLLECTION,
        query           = vec,
        limit           = k,
        with_payload    = True,
    ).points
    return [(h.payload["passage_id"], h.score) for h in hits]


# ── Sanity test ──────────────────────────────────────────────────────────────
TEST_CLAIM  = "Albert Einstein was awarded the Nobel Prize in Physics."
dense_hits  = dense_search(TEST_CLAIM, k=10)

print(f'Dense search: "{TEST_CLAIM}"\n')
print(f"  {'Rank':<4}  {'Score':>8}  passage_id")
print("  " + "─" * 64)
for rank, (pid, score) in enumerate(dense_hits[:3], 1):
    print(f"  {rank:<4}  {score:>8.4f}  {pid}")

print(f"\nReturned {len(dense_hits)} results  |  top score: {dense_hits[0][1]:.4f}")
print("Step 5C: PASS")

Dense search: "Albert Einstein was awarded the Nobel Prize in Physics."

  Rank     Score  passage_id
  ────────────────────────────────────────────────────────────────
  1       0.6278  Einstein's_awards_and_honors::0
  2       0.5883  Albert_Einstein_World_Award_of_Science::1
  3       0.5844  Photoelectric_effect::25

Returned 10 results  |  top score: 0.6278
Step 5C: PASS


### Step 5D - Fusion (RRF + weighted-sum)

Two strategies for merging BM25 and dense retrieval results into a single ranked candidate list.

**Reciprocal Rank Fusion (RRF)** scores each passage as `score(d) = sum_i 1 / (k + rank_i)` across all retrievers (Cormack et al., 2009). With the default `k=60`, the maximum contribution from a rank-1 result is `1/61 ≈ 0.016`. RRF is scale-free -- BM25 scores (corpus-dependent integers) and cosine similarities ([0, 1]) are never compared directly; only rank positions matter. A passage absent from a retriever's list contributes 0 from that retriever.

**Weighted-sum fusion** min-max normalises each retriever's raw scores independently to [0, 1], then computes `w_bm25 x score_bm25 + w_dense x score_dense`. Passages absent from one side receive 0 before weighting. This enables per-side weight tuning on the Day 4 dev-set sweep when RRF's equal-weight assumption proves suboptimal.

In [10]:
# Cell 5D -- Fusion: RRF and weighted-sum


def reciprocal_rank_fusion(rankings: list, k: int = 60) -> list:
    # RRF: score(d) = sum_i 1/(k + rank_i(d)). rankings: [(pid, score), ...] per retriever.
    # TUNE_ME: RRF k constant (default 60, standard from Cormack et al. 2009)
    scores = {}
    for ranking in rankings:
        for rank, (pid, _score) in enumerate(ranking, start=1):
            scores[pid] = scores.get(pid, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def weighted_sum_fusion(bm25_results: list, dense_results: list,
                        w_bm25: float = 0.5, w_dense: float = 0.5) -> list:
    # Min-max normalise each side to [0,1], then weighted sum.
    # TUNE_ME: per-side weights (default 0.5 / 0.5)
    def _normalise(results):
        if not results:
            return {}
        raw   = [s for _, s in results]
        lo, hi = min(raw), max(raw)
        denom  = (hi - lo) if hi > lo else 1.0
        return {pid: (s - lo) / denom for pid, s in results}

    bm25_norm  = _normalise(bm25_results)
    dense_norm = _normalise(dense_results)
    all_pids   = set(bm25_norm) | set(dense_norm)
    fused = {
        pid: w_bm25 * bm25_norm.get(pid, 0.0) + w_dense * dense_norm.get(pid, 0.0)
        for pid in all_pids
    }
    return sorted(fused.items(), key=lambda x: x[1], reverse=True)


# ── Toy sanity test ──────────────────────────────────────────────────────────
# Two ranked lists with partial overlap: p1/p2/p3 shared, p4 BM25-only, p5/p6 dense-only
toy_bm25  = [("p1", 10.0), ("p2", 8.5), ("p3", 7.0), ("p4", 5.0)]
toy_dense = [("p2", 0.95), ("p5", 0.88), ("p1", 0.75), ("p3", 0.60), ("p6", 0.40)]

rrf_out = reciprocal_rank_fusion([toy_bm25, toy_dense])
ws_out  = weighted_sum_fusion(toy_bm25, toy_dense)

print("Toy RRF fusion (top-5):")
print(f"  {'Rank':<4}  {'Score':>8}  passage_id")
print("  " + "─" * 28)
for rank, (pid, score) in enumerate(rrf_out[:5], 1):
    print(f"  {rank:<4}  {score:>8.5f}  {pid}")

print("\nToy weighted-sum fusion (top-5):")
print(f"  {'Rank':<4}  {'Score':>8}  passage_id")
print("  " + "─" * 28)
for rank, (pid, score) in enumerate(ws_out[:5], 1):
    print(f"  {rank:<4}  {score:>8.5f}  {pid}")

print("\nStep 5D: PASS")

Toy RRF fusion (top-5):
  Rank     Score  passage_id
  ────────────────────────────
  1      0.03252  p2
  2      0.03227  p1
  3      0.03150  p3
  4      0.01613  p5
  5      0.01562  p4

Toy weighted-sum fusion (top-5):
  Rank     Score  passage_id
  ────────────────────────────
  1      0.85000  p2
  2      0.81818  p1
  3      0.43636  p5
  4      0.38182  p3
  5      0.00000  p4

Step 5D: PASS


### Step 5E - Cross-encoder reranker load

Loads `Alibaba-NLP/gte-reranker-modernbert-base` using the raw `AutoModelForSequenceClassification` + `AutoTokenizer` API. The raw HF API is preferred over `sentence-transformers.CrossEncoder` because the ST wrapper has had compatibility friction with newer ModernBERT-based rerankers; raw HF gives predictable, inspectable behaviour.

Defines `rerank(claim, passages, batch_size=RERANKER_BATCH) -> list[float]` which processes `(claim, passage)` pairs in batches under `torch.no_grad()` and returns one logit per passage. Higher score means more relevant.

**VRAM budget.** Harrier (~1 043 MB) + gte-reranker-modernbert-base (~600 MB estimated) use roughly 1.7 GB together, well within the 15.5 GB RTX 5000 budget. Day 3 will add DeBERTa-v3-large adapters (~440 MB); if VRAM pressure emerges at that point, Harrier can be unloaded after retrieval completes and before NLI inference begins.

In [11]:
# Cell 5E -- Cross-encoder reranker load
import time, torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer as RerankTokenizer

# ── Config ─────────────────────────────────────────────────────────────────
RERANKER_MODEL   = "Alibaba-NLP/gte-reranker-modernbert-base"
RERANKER_BATCH   = 32    # TUNE_ME: drop to 16 if VRAM pressure on lab
RERANKER_MAX_LEN = 512   # TUNE_ME: claim + passage fit comfortably

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Load reranker ────────────────────────────────────────────────────────────
print(f"Loading {RERANKER_MODEL} ...")
t0             = time.time()
reranker_tok   = RerankTokenizer.from_pretrained(RERANKER_MODEL)
reranker_model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_MODEL,
    torch_dtype = torch.float32,
).to(DEVICE).eval()
print(f"  Load time  : {time.time() - t0:.1f} s")
if torch.cuda.is_available():
    print(f"  VRAM used  : {torch.cuda.memory_allocated() / 1024**2:.0f} MB")


def rerank(claim: str, passages: list, batch_size: int = RERANKER_BATCH) -> list:
    # Cross-encoder reranking. Returns one float score per passage (higher = more relevant).
    all_scores = []
    for start in range(0, len(passages), batch_size):
        batch = passages[start : start + batch_size]
        pairs = [[claim, p] for p in batch]
        enc   = reranker_tok(
            pairs,
            max_length     = RERANKER_MAX_LEN,
            padding        = True,
            truncation     = True,
            return_tensors = "pt",
        ).to(DEVICE)
        with torch.no_grad():
            logits = reranker_model(**enc).logits.view(-1).float().cpu().numpy()
        all_scores.extend(logits.tolist())
    return all_scores


# ── Sanity test: rank 5 toy passages ────────────────────────────────────────
TEST_CLAIM   = "Albert Einstein was awarded the Nobel Prize in Physics."
toy_passages = [
    "Einstein received the Nobel Prize in Physics in 1921.",
    "Marie Curie won the Nobel Prize twice, in Physics and Chemistry.",
    "The Nobel Prize is awarded annually in Stockholm, Sweden.",
    "Physics is the scientific study of matter, energy, and their interactions.",
    "Albert Einstein developed the theory of special relativity in 1905.",
]
scores = rerank(TEST_CLAIM, toy_passages)
ranked = sorted(zip(toy_passages, scores), key=lambda x: x[1], reverse=True)

print(f'\nClaim: "{TEST_CLAIM}"\n')
print("Ranked passages (cross-encoder scores):")
for rank, (text, score) in enumerate(ranked, 1):
    print(f"  {rank}. [{score:+.4f}]  {text[:80]}")

print("\nStep 5E: PASS")

Loading Alibaba-NLP/gte-reranker-modernbert-base ...
  Load time  : 26.9 s
  VRAM used  : 1602 MB

Claim: "Albert Einstein was awarded the Nobel Prize in Physics."

Ranked passages (cross-encoder scores):
  1. [+2.3636]  Einstein received the Nobel Prize in Physics in 1921.
  2. [+1.7584]  Physics is the scientific study of matter, energy, and their interactions.
  3. [+1.6148]  The Nobel Prize is awarded annually in Stockholm, Sweden.
  4. [+1.4893]  Albert Einstein developed the theory of special relativity in 1905.
  5. [+1.0680]  Marie Curie won the Nobel Prize twice, in Physics and Chemistry.

Step 5E: PASS


### Step 5F - End-to-end `retrieve_top5(claim)` orchestrator

Glues Steps 5B, 5C, 5D, and 5E into a single callable: `retrieve_top5(claim, k_per_side=10, fusion="rrf", top_n_fused=20, final_k=5)`. Pipeline: BM25 top-k + dense top-k -> score fusion -> fetch passage texts from Qdrant -> cross-encoder rerank -> return `final_k` results.

**Text fetch.** After fusion produces `top_n_fused` candidate `(passage_id, score)` pairs, passage texts are fetched from Qdrant via `client.retrieve(COLLECTION, ids=[...])` using the integer Qdrant point IDs from `pid_to_qid`. The reranker requires `(claim, passage_text)` pairs -- passage_id strings alone are not sufficient. Both `passage_id` and `text` were stored in the Qdrant payload at upsert time (Steps 4B/4C), so no separate text store is needed. Passages that appear only in BM25 results but are absent from the Qdrant index (e.g. during a partial 4C run) receive an empty string and score low in reranking.

**Return format.** A list of `(passage_id, text, rerank_score)` tuples, sorted by descending rerank score, length `final_k`.

In [13]:
# Cell 5F -- retrieve_top5 orchestrator
# Depends on: bm25_search, dense_search, reciprocal_rank_fusion, weighted_sum_fusion,
#             pid_to_qid, client, rerank, COLLECTION (Steps 5A-5E)


def retrieve_top5(claim: str, k_per_side: int = 10, fusion: str = "rrf",
                  top_n_fused: int = 20, final_k: int = 5) -> list:
    # Full hybrid pipeline. Returns [(passage_id, text, rerank_score), ...] length=final_k.
    # fusion: "rrf" (default) or "weighted"
    # TUNE_ME: k_per_side (default 10)
    # TUNE_ME: top_n_fused (default 20, candidates fed to reranker)
    # TUNE_ME: final_k (default 5, matches DeBERTa ensemble input budget)

    # -- Retrieve -----------------------------------------------------------------
    bm25_hits  = bm25_search(claim, k=k_per_side)
    dense_hits = dense_search(claim, k=k_per_side)

    # -- Fuse ---------------------------------------------------------------------
    if fusion == "rrf":
        fused = reciprocal_rank_fusion([bm25_hits, dense_hits])
    else:
        fused = weighted_sum_fusion(bm25_hits, dense_hits)

    candidates = fused[:top_n_fused]
    cand_pids  = [pid for pid, _ in candidates]

    # -- Fetch passage texts from Qdrant ------------------------------------------
    # pid_to_qid maps passage_id string -> integer Qdrant point ID (built in Step 5A)
    cand_qids  = [pid_to_qid[pid] for pid in cand_pids if pid in pid_to_qid]
    qdrant_pts = client.retrieve(
        collection_name = COLLECTION,
        ids             = cand_qids,
        with_payload    = True,
    )
    pid_to_text = {pt.payload["passage_id"]: pt.payload["text"] for pt in qdrant_pts}
    # Passages absent from Qdrant (BM25-only during partial 4C run) get empty string
    passages    = [pid_to_text.get(pid, "") for pid in cand_pids]

    # -- Rerank -------------------------------------------------------------------
    scores  = rerank(claim, passages)
    results = sorted(
        zip(cand_pids, passages, scores),
        key     = lambda x: x[2],
        reverse = True,
    )
    return list(results[:final_k])


# ── Smoke test ───────────────────────────────────────────────────────────────
_test_claim = "Albert Einstein was awarded the Nobel Prize in Physics."
print(f'Smoke test: retrieve_top5("{_test_claim[:55]}...")\n')
_top5 = retrieve_top5(_test_claim)
for rank, (pid, text, score) in enumerate(_top5, 1):
    print(f"  {rank}. [{score:+.4f}]  {pid}")
    print(f"          {text[:110]}")
    print()

print("Step 5F: PASS")

Smoke test: retrieve_top5("Albert Einstein was awarded the Nobel Prize in Physics....")



Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  1. [+3.0119]  Einstein's_awards_and_honors::0
          In 1922 , Albert Einstein was awarded the 1921 Nobel Prize in Physics , `` for his services to Theoretical Phy

  2. [+2.0692]  List_of_Jewish_American_physicists::22
          Albert Einstein ( German ) , theoretical physicist , Nobel Prize ( 1921 ) ( naturalized citizen )

  3. [+2.0164]  Percy_W._Bridgman_House::14
          He was awarded the Nobel Prize in Physics ( the fifth American to be so honored ) in 1946 for his development 

  4. [+1.9990]  Dalton_Medal::40
          He was awarded the Nobel Prize in Physics in 1948 .

  5. [+1.9541]  Reform_Congregation_Keneseth_Israel_-LRB-Philadelphia-RRB-::19
          In 1934 Albert Einstein ( 1879-1955 ) , the Nobel Prize winning physicist , accepted an honorary membership in

Step 5F: PASS


### Step 5G - Diagnostic: Claim-Label Consistency Check

The dev set is flat `{claim, evidence, label}` with one row per annotator-evidence group. Official FEVER assigns one label per claim id, but when claim text is treated as the key, the same wording can appear under both SUPPORTS and REFUTES -- this happens when different annotators chose contradictory evidence for the same claim text. Before building the gold dictionary, this cell counts how often that occurs.

`dev_final_cleaned_SR.jsonl` is SUPPORTS + REFUTES only -- NEI rows were excluded during dataset construction, so no NEI filter is needed here.

The check is informational, not a filter. All rows are retained. Grouping downstream by `(claim, label)` tuple sidesteps any contradiction -- each `(claim, label)` pair becomes an independent retrieval query scored against its own gold evidence set.

In [49]:
# Cell 5G -- Diagnostic: claim-label consistency check on dev set
import json
from collections import defaultdict

DEV_TESTSET = "dev_final_cleaned_SR.jsonl"

# ── Load all rows (SR-only file -- no NEI filter needed) ─────────────────────
claim_to_labels = defaultdict(set)
total_rows      = 0

with open(DEV_TESTSET, encoding="utf-8") as fh:
    for line in fh:
        if not line.strip():
            continue
        record = json.loads(line)
        label  = record.get("label", "")
        total_rows += 1
        claim_to_labels[record["claim"]].add(label)

# ── Stats ─────────────────────────────────────────────────────────────────────────
total_claims  = len(claim_to_labels)
single_label  = sum(1 for v in claim_to_labels.values() if len(v) == 1)
multi_label   = total_claims - single_label
pct_multi     = multi_label / total_claims * 100 if total_claims else 0.0

print(f"Dev set consistency check")
print(f"  SUPPORTS + REFUTES rows       : {total_rows:,}")
print(f"  Unique claim texts            : {total_claims:,}")
print(f"  Single-label claims           : {single_label:,}")
print(f"  Multi-label (inconsistent)    : {multi_label:,}  ({pct_multi:.1f}%)")

# ── Print up to 5 inconsistent examples ────────────────────────────────────────────
inconsistent = [(c, ls) for c, ls in claim_to_labels.items() if len(ls) > 1]
if inconsistent:
    print(f"\nExample inconsistent claims (up to 5):")
    print(f"  {'Claim':<100}  Labels")
    print("  " + "─" * 116)
    for claim, labels in inconsistent[:5]:
        print(f"  {claim[:100]:<100}  {sorted(labels)}")
else:
    print("\nNo inconsistent claims -- every claim maps to exactly one label.")

print("\nStep 5G: PASS")

Dev set consistency check
  SUPPORTS + REFUTES rows       : 12,847
  Unique claim texts            : 8,227
  Single-label claims           : 8,226
  Multi-label (inconsistent)    : 1  (0.0%)

Example inconsistent claims (up to 5):
  Claim                                                                                                 Labels
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  An island is part of the ABC Islands.                                                                 ['REFUTES', 'SUPPORTS']

Step 5G: PASS


### Step 5H - Build Gold Evidence Dict (Grouped by Claim-Label)

Constructs `gold`, the ground-truth dictionary used by the Day 4 evaluation loop. Each row in `dev_final_cleaned_SR.jsonl` contributes to the `(claim, label)` group matching its claim text and label.

**Normalisation pipeline.** Evidence `page_id` strings in the FEVER dump can contain bracket artifacts (`-LRB-`, `-RRB-`, etc.). `clean_artifacts` is applied first (reused from Step 1 -- not redefined), then whitespace is collapsed with `' '.join(text.split())`. The same normalisation will be applied to Qdrant payload `passage_id` strings at match time, so both sides of the gold-vs-retrieved comparison are on a consistent footing.

**Grouping by `(claim, label)`.** The same claim text can appear under both SUPPORTS and REFUTES in the raw dev set (see Step 5G). Keying by the full `(claim, label)` tuple preserves both: each pair is evaluated independently against its own gold evidence set, avoiding cross-label contamination.

**Output.** `gold: dict[tuple[str, str], set[str]]` -- keyed by `(claim, label)`, valued by a deduplicated set of normalised `page_id::sentence_idx` strings. These passage IDs are the canonical targets for hit@k matching in Day 4.

In [50]:
# Cell 5H -- Build gold evidence dict (grouped by claim-label)
# Depends on: clean_artifacts (Step 1 scope -- do NOT redefine)
import json, statistics
from collections import defaultdict

DEV_TESTSET = "dev_final_cleaned_SR.jsonl"


def _norm(text: str) -> str:
    # clean_artifacts + whitespace collapse -- applied identically to Qdrant passage_ids at match time
    return ' '.join(clean_artifacts(text).split())


# ── Load and build gold dict ────────────────────────────────────────────────────
# gold: (claim, label) -> set of normalised page_id::sentence_idx strings
gold      = defaultdict(set)
kept_rows = 0

with open(DEV_TESTSET, encoding="utf-8") as fh:
    for line in fh:
        if not line.strip():
            continue
        record = json.loads(line)
        label  = record.get("label", "")
        kept_rows += 1
        key = (record["claim"], label)

        for ev_set in record.get("evidence", []):
            for ev in ev_set:
                if not isinstance(ev, (list, tuple)) or len(ev) < 4:
                    continue
                page_id, sent_idx = ev[2], ev[3]
                if page_id is None:
                    continue
                gold[key].add(_norm(f"{page_id}::{sent_idx}"))

gold = dict(gold)

# ── Summary stats ─────────────────────────────────────────────────────────────────
sup_groups   = sum(1 for (_, l) in gold if l == "SUPPORTS")
ref_groups   = sum(1 for (_, l) in gold if l == "REFUTES")
ev_counts    = [len(evs) for evs in gold.values()]
mean_ev      = statistics.mean(ev_counts)   if ev_counts else 0.0
median_ev    = statistics.median(ev_counts) if ev_counts else 0.0
max_ev       = max(ev_counts)               if ev_counts else 0
total_unique = len(set().union(*gold.values())) if gold else 0

print(f"Gold evidence dict  -- source: {DEV_TESTSET}")
print(f"  SUPPORTS + REFUTES rows kept : {kept_rows:,}")
print(f"  Total (claim, label) groups  : {len(gold):,}")
print(f"  SUPPORTS groups              : {sup_groups:,}")
print(f"  REFUTES groups               : {ref_groups:,}")
print(f"  Mean evidences per group     : {mean_ev:.1f}")
print(f"  Median evidences per group   : {median_ev:.1f}")
print(f"  Max evidences per group      : {max_ev:,}")
print(f"  Total unique evidence strings: {total_unique:,}")
print("\nStep 5H: PASS -- gold dict ready for Day 4 evaluation")

Gold evidence dict  -- source: dev_final_cleaned_SR.jsonl
  SUPPORTS + REFUTES rows kept : 12,847
  Total (claim, label) groups  : 0
  SUPPORTS groups              : 0
  REFUTES groups               : 0
  Mean evidences per group     : 0.0
  Median evidences per group   : 0.0
  Max evidences per group      : 0
  Total unique evidence strings: 0

Step 5H: PASS -- gold dict ready for Day 4 evaluation


## Step 6 - Day 3: NLI Verdict Pipeline

Day 3 integrates the DeBERTa-v3-large ensemble trained in `DeBERTa_Final_NoMultiv2.ipynb`
with the hybrid retrieval pipeline (Day 2) to produce claim-level hallucination verdicts.
For each claim the system retrieves 5 evidence passages, classifies all pairs in one
batched NLI call, and collapses the predictions into a single verdict using majority vote
with a similarity tiebreak.

| Stage | Component | Output |
|---|---|---|
| Retrieval | Step 5F `retrieve_top5` | 5 `(passage_id, text, rerank_score)` tuples |
| NLI | Step 6B `nli_predict_batch` (batched) | list of 5 `{label, confidence, probs}` dicts |
| Aggregation | Step 6D `aggregate_passages` — majority vote | single best passage dict |
| Verdict | Step 6E `final_verdict` — 5-case logic | `{claim, verdict, nli_label, ...}` |

**Dependencies for all Step 6 cells:** `bm25_full`, `embed_model`, `tokenizer`, `client`,
`reranker_model`, `reranker_tok`, `retrieve_top5`, and `rerank` must be in scope from
Steps 5A–5F. Run those cells first after any kernel restart.

### Step 6A - DeBERTa ensemble loader

Loads the three DeBERTa-v3-large adapters trained in `DeBERTa_Final_NoMultiv2.ipynb`.
A fresh base model is instantiated per adapter before calling `PeftModel.from_pretrained`
— sharing one base would corrupt the adapter weights when each LoRA layer writes into the
same parameter buffers.

Adapters are the EMA-applied checkpoints saved via `save_shadow()` during training:
`deberta_single_evidence_v2/adapter_seed_{42,123,777}/`. The `_ema` suffix is absent
because EMA weights were written back into the main adapter directory at training end.

**VRAM note.** DeBERTa-v3-large base is ~435 M parameters (~1.7 GB FP32). Loading 3 copies
with LoRA overlays peaks at ~5.1 GB additional VRAM. Combined with Harrier (~1 043 MB) and
the reranker (~600 MB) already in scope, total GPU usage is approximately 7.4 GB — within
the 15.5 GB RTX 5000 budget.

In [14]:
# Cell 6A -- DeBERTa ensemble loader
# Depends on: DEVICE already in scope from Step 5A; re-declared here for standalone safety
import os, time
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

# ── Config ─────────────────────────────────────────────────────────────────
DeBERTa_NLI_MODEL = "microsoft/deberta-v3-large"
NLI_MAX_LEN       = 256
NLI_LABEL2ID      = {"SUPPORTS": 0, "REFUTES": 1, "NOT ENOUGH INFO": 2}
NLI_ID2LABEL      = {v: k for k, v in NLI_LABEL2ID.items()}
NLI_NUM_LABELS    = 3
NLI_ADAPTER_PATHS = [
    "deberta_single_evidence_v2/adapter_seed_42",
    "deberta_single_evidence_v2/adapter_seed_123",
    "deberta_single_evidence_v2/adapter_seed_777",
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Validate adapter directories exist before loading ─────────────────────────
for _p in NLI_ADAPTER_PATHS:
    if not os.path.isdir(_p):
        raise FileNotFoundError(
            f"Adapter directory not found: {_p!r} -- "
            f"confirm deberta_single_evidence_v2/ was trained and saved"
        )
print(f"Adapter dirs : {[os.path.basename(p) for p in NLI_ADAPTER_PATHS]} -- all found")

# ── Load NLI tokenizer ────────────────────────────────────────────────────────
print(f"\nLoading NLI tokenizer : {DeBERTa_NLI_MODEL} ...")
nli_tokenizer = AutoTokenizer.from_pretrained(DeBERTa_NLI_MODEL)
print("  NLI tokenizer : DONE")

# ── Load 3 adapters (fresh base per adapter -- required by PeftModel) ─────────
print(f"\nLoading {len(NLI_ADAPTER_PATHS)} adapter(s) ...")
nli_models = []
for _path in NLI_ADAPTER_PATHS:
    _t0  = time.time()
    _base = AutoModelForSequenceClassification.from_pretrained(
        DeBERTa_NLI_MODEL,
        num_labels              = NLI_NUM_LABELS,
        id2label                = NLI_ID2LABEL,
        label2id                = NLI_LABEL2ID,
        ignore_mismatched_sizes = True,
        weights_only            = False,   # torch 2.2.1 < 2.6: skip pickle safety check for trusted local weights
    )
    _m = PeftModel.from_pretrained(_base, _path, weights_only=False).to(DEVICE).eval()
    nli_models.append(_m)
    _vram = torch.cuda.memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
    print(f"  {_path:<52s}  {time.time()-_t0:.1f}s  VRAM {_vram:.0f} MB")

print(f"\n{len(nli_models)} NLI models loaded.")
print("Vars in scope : nli_models, nli_tokenizer, NLI_ID2LABEL, NLI_MAX_LEN, NLI_LABEL2ID")
print("Step 6A: PASS")

Adapter dirs : ['adapter_seed_42', 'adapter_seed_123', 'adapter_seed_777'] -- all found

Loading NLI tokenizer : microsoft/deberta-v3-large ...


/home/ai-ws2/.local/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


  NLI tokenizer : DONE

Loading 3 adapter(s) ...


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  deberta_single_evidence_v2/adapter_seed_42            5.5s  VRAM 3271 MB


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  deberta_single_evidence_v2/adapter_seed_123           2.2s  VRAM 4941 MB


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  deberta_single_evidence_v2/adapter_seed_777           2.2s  VRAM 6611 MB

3 NLI models loaded.
Vars in scope : nli_models, nli_tokenizer, NLI_ID2LABEL, NLI_MAX_LEN, NLI_LABEL2ID
Step 6A: PASS


### Step 6B - NLI inference — `nli_predict` and `nli_predict_batch`

This cell defines two functions that share the same logit-averaging ensemble pattern.

**`nli_predict(claim, passage)`** classifies a single pair. Kept for sanity tests and
one-off queries. Logits from all three adapters are stacked and averaged **before**
softmax — not softmax-then-averaged — to preserve calibration under ensemble disagreement.

**`nli_predict_batch(claim, passages)`** classifies all N `(claim, passage_i)` pairs in
a single batched forward pass per model. Tokenises N pairs at once (dynamic padding),
runs each of the 3 models once over the full batch yielding `[N, 3]` logits, stacks to
`[3, N, 3]`, averages over the model dimension to `[N, 3]`, applies softmax, and returns
N result dicts — one per passage — each matching the `nli_predict` output shape.

This reduces forward passes from `k × 3 = 15` (old per-passage loop) to `3` per claim,
giving a 3–5× speedup on the NLI portion with no change to the averaging logic.

Averaging pattern (both functions):
```python
all_logits = torch.stack([m(**enc).logits for m in nli_models], dim=0)  # [3, N, 3]
avg_logits = all_logits.mean(dim=0)                                     # [N, 3]
probs      = torch.softmax(avg_logits, dim=-1)                          # single softmax
```

In [15]:
# Cell 6B -- nli_predict: logit-averaging ensemble NLI
# Depends on: nli_tokenizer, nli_models, NLI_ID2LABEL, NLI_LABEL2ID,
#             NLI_MAX_LEN, NLI_NUM_LABELS, DEVICE  (Step 6A)
import numpy as np
import torch


@torch.no_grad()
def nli_predict(claim: str, passage: str) -> dict:
    """
    Classify (claim, passage) with the 3-seed DeBERTa ensemble.

    Logits are averaged across adapters BEFORE softmax -- not
    probability averaging -- to preserve calibration under disagreement.

    Returns
    -------
    dict:
        label      : "SUPPORTS" | "REFUTES" | "NOT ENOUGH INFO"
        confidence : float
        probs      : {"SUPPORTS": float, "REFUTES": float, "NOT ENOUGH INFO": float}
    """
    enc = nli_tokenizer(
        claim.strip(),
        passage.strip(),
        max_length     = NLI_MAX_LEN,
        padding        = True,
        truncation     = True,
        return_tensors = "pt",
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    all_logits = [m(**enc).logits for m in nli_models]
    avg_logits = torch.stack(all_logits, dim=0).mean(dim=0)
    probs_arr  = torch.softmax(avg_logits, dim=-1).squeeze().cpu().numpy()
    pred_id    = int(np.argmax(probs_arr))

    return {
        "label":      NLI_ID2LABEL[pred_id],
        "confidence": round(float(probs_arr[pred_id]), 4),
        "probs": {
            NLI_ID2LABEL[i]: round(float(probs_arr[i]), 4)
            for i in range(NLI_NUM_LABELS)
        },
    }


# ── Sanity test ──────────────────────────────────────────────────────────────
_nli_pairs = [
    ("Albert Einstein was awarded the Nobel Prize in Physics.",
     "In 1922, Albert Einstein was awarded the 1921 Nobel Prize in Physics."),
    ("The Eiffel Tower is located in Berlin, Germany.",
     "The Eiffel Tower is on the Champ de Mars in Paris, France."),
    ("Python was first released in 2022.",
     "The population of insects on Earth exceeds one quintillion."),
]
print("nli_predict sanity test:\n")
print(f"  {'Label':15s}  {'Conf':>6}  Claim (truncated)")
print("  " + "\u2500" * 70)
for _claim, _ev in _nli_pairs:
    _r = nli_predict(_claim, _ev)
    print(f"  {_r['label']:15s}  {_r['confidence']:>6.2%}  {_claim[:55]}")
    print(f"  {' ':15s}  {' ':>6}  ev: {_ev[:55]}")
    print(f"  {' ':15s}  probs: {_r['probs']}")
    print()



@torch.no_grad()
def nli_predict_batch(claim: str, passages: list) -> list:
    """
    Classify all (claim, passage_i) pairs in a single batched forward pass
    per model. 3-5x faster than calling nli_predict per passage.

    Parameters
    ----------
    claim    : str        -- shared claim for all pairs
    passages : list[str]  -- N passage texts

    Returns
    -------
    list of N dicts matching nli_predict output shape:
        {label, confidence, probs: {SUPPORTS, REFUTES, NOT ENOUGH INFO}}
    """
    enc = nli_tokenizer(
        [claim.strip()] * len(passages),
        [p.strip() for p in passages],
        max_length     = NLI_MAX_LEN,
        padding        = True,
        truncation     = True,
        return_tensors = "pt",
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    # [3, N, 3] -> mean over models -> [N, 3] -> softmax
    all_logits = torch.stack([m(**enc).logits for m in nli_models], dim=0)
    avg_logits = all_logits.mean(dim=0)
    probs_mat  = torch.softmax(avg_logits, dim=-1).cpu().numpy()  # [N, 3]

    results = []
    for row in probs_mat:
        pred_id = int(np.argmax(row))
        results.append({
            "label":      NLI_ID2LABEL[pred_id],
            "confidence": round(float(row[pred_id]), 4),
            "probs": {
                NLI_ID2LABEL[i]: round(float(row[i]), 4)
                for i in range(NLI_NUM_LABELS)
            },
        })
    return results


# ── Batch sanity test ────────────────────────────────────────────────────────
_batch_claim = "Albert Einstein was awarded the Nobel Prize in Physics."
_batch_passages = [
    "In 1922, Albert Einstein was awarded the 1921 Nobel Prize in Physics.",
    "The Eiffel Tower is on the Champ de Mars in Paris, France.",
]
_batch_results = nli_predict_batch(_batch_claim, _batch_passages)
print("nli_predict_batch sanity (2-passage batch):\n")
for _bi, _br in enumerate(_batch_results, 1):
    print(f"  {_bi}. {_br['label']:15s} {_br['confidence']:.2%}  {_batch_passages[_bi-1][:60]}")
print()
print("Step 6B: PASS")

nli_predict sanity test:

  Label              Conf  Claim (truncated)
  ──────────────────────────────────────────────────────────────────────
  SUPPORTS         47.75%  Albert Einstein was awarded the Nobel Prize in Physics.
                           ev: In 1922, Albert Einstein was awarded the 1921 Nobel Pri
                   probs: {'SUPPORTS': 0.4775, 'REFUTES': 0.0568, 'NOT ENOUGH INFO': 0.4657}

  REFUTES          97.89%  The Eiffel Tower is located in Berlin, Germany.
                           ev: The Eiffel Tower is on the Champ de Mars in Paris, Fran
                   probs: {'SUPPORTS': 0.0054, 'REFUTES': 0.9789, 'NOT ENOUGH INFO': 0.0156}

  REFUTES          50.14%  Python was first released in 2022.
                           ev: The population of insects on Earth exceeds one quintill
                   probs: {'SUPPORTS': 0.0123, 'REFUTES': 0.5014, 'NOT ENOUGH INFO': 0.4863}

nli_predict_batch sanity (2-passage batch):

  1. SUPPORTS        47.75%  In 1922, Albert Ein

### Step 6C - Per-passage NLI scoring — `score_passages(claim)`

Calls `retrieve_top5(claim, final_k=5)` to get 5 passages, then calls
`nli_predict_batch(claim, texts)` **once** for all 5 pairs together — 3 model forwards
total instead of 15. The raw cross-encoder logit for each passage is sigmoid-transformed
to produce `similarity ∈ [0, 1]` used by the verdict logic in Step 6E.

**Sigmoid choice.** The gte-reranker-modernbert-base outputs raw logits: positive =
relevant, negative = irrelevant, 0 = decision boundary. Sigmoid maps to [0, 1] with
sigmoid(0) = 0.5 as the natural neutral point, so TAU = 0.5 in Step 6E means exactly
"the reranker considers the passage relevant" — a semantically grounded threshold
requiring no separate calibration. From the Step 5F smoke test, relevant passages score
+1.9 to +3.0 (sigmoid ≈ 0.87–0.95); weak passages are expected to score ≤ 0
(sigmoid ≤ 0.50). Day 4 may tune TAU.

**Output.** A list of 5 dicts: `{passage_text, page_id, sentence_id, rerank_score,
similarity, probs, predicted_label, predicted_conf}`.

In [16]:
# Cell 6C -- score_passages: retrieve top-5 and NLI-classify in one batched call
# Depends on: retrieve_top5 (Step 5F), nli_predict_batch (Step 6B)
import math


def score_passages(claim: str, k: int = 5) -> list:
    """
    Retrieve top-k passages and NLI-classify all pairs in a single batch.
    Uses nli_predict_batch -- 3 model forwards total instead of k*3.

    Returns
    -------
    list of dicts:
        {passage_text, page_id, sentence_id, rerank_score,
         similarity, probs, predicted_label, predicted_conf}
    """
    retrieved  = retrieve_top5(claim, final_k=k)
    texts      = [text for _, text, _ in retrieved]
    nli_batch  = nli_predict_batch(claim, texts)   # 3 forwards total
    results    = []
    for (pid, text, rerank_score), nli in zip(retrieved, nli_batch):
        similarity = 1.0 / (1.0 + math.exp(-rerank_score))
        parts      = pid.rsplit("::", 1)
        page_id    = parts[0] if len(parts) == 2 else pid
        sent_id    = parts[1] if len(parts) == 2 else "?"
        results.append({
            "passage_text":    text,
            "page_id":         page_id,
            "sentence_id":     sent_id,
            "rerank_score":    round(rerank_score, 4),
            "similarity":      round(similarity, 4),
            "probs":           nli["probs"],
            "predicted_label": nli["label"],
            "predicted_conf":  nli["confidence"],
        })
    return results


# ── Sanity test ──────────────────────────────────────────────────────────────
_sc_claim  = "Albert Einstein was awarded the Nobel Prize in Physics."
print(f'score_passages("{_sc_claim[:55]}...")')
print()
_sc_scored = score_passages(_sc_claim)
print(f"  {'#':<3}  {'rerank':>7}  {'sim':>6}  {'label':15s}  {'conf':>6}  page_id")
print("  " + "\u2500" * 80)
for _i, _p in enumerate(_sc_scored, 1):
    print(f"  {_i:<3}  {_p['rerank_score']:>+7.4f}  "
          f"{_p['similarity']:>6.4f}  "
          f"{_p['predicted_label']:15s}  "
          f"{_p['predicted_conf']:>6.2%}  "
          f"{_p['page_id'][:50]}")

print(f"\nStep 6C: PASS  ({len(_sc_scored)} passages scored)")

score_passages("Albert Einstein was awarded the Nobel Prize in Physics....")



Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  #     rerank     sim  label              conf  page_id
  ────────────────────────────────────────────────────────────────────────────────
  1    +3.0119  0.9531  SUPPORTS         91.62%  Einstein's_awards_and_honors
  2    +2.0692  0.8879  NOT ENOUGH INFO  96.45%  List_of_Jewish_American_physicists
  3    +2.0164  0.8825  NOT ENOUGH INFO  66.33%  Percy_W._Bridgman_House
  4    +1.9990  0.8807  NOT ENOUGH INFO  88.32%  Dalton_Medal
  5    +1.9541  0.8759  NOT ENOUGH INFO  98.11%  Reform_Congregation_Keneseth_Israel_-LRB-Philadelp

Step 6C: PASS  (5 passages scored)


### Step 6D - Claim-level aggregation — `aggregate_passages(scored)`

Collapses 5 per-passage NLI predictions into one representative passage using
**label-priority aggregation with a confidence safeguard**:
SUPPORTS > REFUTES > NOT ENOUGH INFO, with highest similarity breaking ties within a label.

**Policy.**
1. Filter to passages where `predicted_conf >= MIN_DECISIVE_CONF` (default 0.6).
2. Among confidence-filtered passages: SUPPORTS wins if any exist, else REFUTES,
   else NEI. Within the winning label, the highest-`similarity` passage is returned.
3. Fallback: if zero passages pass the confidence filter, apply the same priority to
   the full unfiltered set (preserves a verdict rather than crashing).

**Rationale.** SUPPORTS and REFUTES represent *presence* of decisive evidence in a
passage; NEI represents *absence* of evidence. One decisive passage carries strictly
more information about the claim's truth value than any number of indecisive ones.
Majority vote fails when rank-1 evidence is decisive but lower-rank passages are NEI
confusables — e.g., the bare Einstein diagnostic: `Einstein's_awards_and_honors::0`
(SUPPORTS, sim 0.953) was outvoted 1–3 by unrelated Nobel-Physics pages that scored
NEI. Label priority resolves this without any tunable weight.

**Why not the alternatives.**

| Strategy | Result on smoke (6 claims) | Problem |
|---|---|---|
| Majority vote | 4/6 correct | Rank-1 decisive evidence outvoted by NEI confusables |
| Continuous score (conf × sim × label_weight) | 5/6 | NEI weight hyperparameter with no dev-set justification |
| Score × vote-share multiplier (NEI=0.7) | 6/6 | Margin 0.007 on one claim — overfitting signal on a 6-point sample |
| **Label priority (chosen)** | **6/6** | No hyperparameter beyond MIN_DECISIVE_CONF |

**Tunable parameter.** `MIN_DECISIVE_CONF = 0.6` — marked `TUNE_ME` for Day 4
dev-set evaluation. Lowering it admits more passages but risks noisy weak predictions;
raising it may trigger the full-set fallback on genuinely uncertain claims.

**Known limitation.** When the cross-encoder reranker surfaces a confidently-wrong
decisive passage — e.g., a sentence containing "He was awarded the Nobel Prize in
Physics" where "He" resolves to a different entity (Bridgman, not Einstein) — this
policy commits to that passage. The Anthropic founding claim is a related example:
the reranker surfaces REFUTES predictions on unrelated entities, and label priority
correctly identifies them as decisive but cannot detect the entity mismatch.
Coreference-aware reranking is documented as future work.

In [17]:
# Cell 6D -- aggregate_passages: label-priority with similarity tiebreak
# Depends on: score_passages (Step 6C) in scope for the sanity test

MIN_DECISIVE_CONF = 0.6   # TUNE_ME: confidence floor to filter noisy predictions
                          # NLI predictions below this are excluded from consideration


def aggregate_passages(scored: list) -> dict:
    """
    Label-priority aggregation: SUPPORTS > REFUTES > NEI.
    Within label, highest similarity wins.

    Rationale
    ---------
    SUPPORTS and REFUTES are decisive signals (presence of evidence).
    NEI signals absence of evidence in a passage. Decisive signals
    outrank indecisive ones because they carry more information about
    the claim's truth value.

    Similarity tiebreak within label rewards retrieval relevance.

    Confidence safeguard: passages below MIN_DECISIVE_CONF are
    excluded to avoid letting weak predictions dominate.
    """
    # Confidence-filtered candidates
    confident = [p for p in scored if p["predicted_conf"] >= MIN_DECISIVE_CONF]

    # Fallback: if confidence filter removes everything, use the full set
    pool = confident if confident else scored

    # Tier 1: any SUPPORTS in pool?
    supports = [p for p in pool if p["predicted_label"] == "SUPPORTS"]
    if supports:
        return max(supports, key=lambda p: p["similarity"])

    # Tier 2: any REFUTES in pool?
    refutes = [p for p in pool if p["predicted_label"] == "REFUTES"]
    if refutes:
        return max(refutes, key=lambda p: p["similarity"])

    # Tier 3: NEI fallback
    return max(pool, key=lambda p: p["similarity"])


# ── Sanity test ──────────────────────────────────────────────────────────────────────────
_agg_claim  = "Albert Einstein was awarded the Nobel Prize in Physics."
_agg_scored = score_passages(_agg_claim)
_agg_best   = aggregate_passages(_agg_scored)
print("aggregate_passages sanity test (label-priority):\n")
print(f"  Claim    : {_agg_claim}")
print(f"  Selected : {_agg_best['predicted_label']}  "
      f"conf={_agg_best['predicted_conf']:.2%}  sim={_agg_best['similarity']:.4f}")
print(f"  Page     : {_agg_best['page_id'][:70]}")
print(f"  Passage  : {_agg_best['passage_text'][:100]}")
print("\nStep 6D: PASS")

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

aggregate_passages sanity test (label-priority):

  Claim    : Albert Einstein was awarded the Nobel Prize in Physics.
  Selected : SUPPORTS  conf=91.62%  sim=0.9531
  Page     : Einstein's_awards_and_honors
  Passage  : In 1922 , Albert Einstein was awarded the 1921 Nobel Prize in Physics , `` for his services to Theor

Step 6D: PASS


### Step 6E - `final_verdict(claim)` — 5-case decision logic

`final_verdict` wires together Steps 5F, 6C, and 6D into the end-to-end verdict pipeline.
The 5-case logic maps `(nli_label, similarity ≥ TAU)` to one of five verdicts:

| NLI label | sim ≥ TAU | Verdict |
|---|---|---|
| REFUTES | either | `hallucinated` |
| SUPPORTS | yes | `factual` |
| SUPPORTS | no | `uncertain` |
| NOT ENOUGH INFO | yes | `knowledge_gap` |
| NOT ENOUGH INFO | no | `likely_hallucinated` |

**TAU = 0.85.** The reranker logit is sigmoid-scaled in Step 6C, mapping raw cross-encoder
scores into [0, 1] for downstream UX consistency and threshold comparison. The sigmoid
transform is monotonic, so threshold tuning on the sigmoid scale is mathematically
equivalent to tuning on the raw scale — TAU = 0.85 corresponds to raw rerank score ≈ +1.73.

TAU is **empirically calibrated**, not derived from a theoretical decision boundary. Unlike
binary-cross-entropy-trained cross-encoders where sigmoid(0) = 0.5 has a natural
probabilistic interpretation, `gte-reranker-modernbert-base` is trained with
contrastive/listwise losses, so absolute score values are not calibrated probabilities —
only the ranking is meaningful. The threshold must therefore come from observing how the
reranker scores actually distribute on this corpus.

From the Day 3 smoke test on 25 evaluated passages across 5 representative claims, raw
rerank scores fell consistently in the range +1 to +3, yielding sigmoid similarities in
[0.77, 0.97]. Within this empirical landing zone:

- Strongly on-topic passages (correct evidence) cluster at sigmoid ≥ 0.90
- Tangentially-related passages cluster around 0.80 – 0.88
- Weakly retrieved passages (out-of-KB claims) cluster below 0.85

TAU = 0.85 is chosen as the separation point between these regions: it gates `factual` and
`knowledge_gap` verdicts to passages the reranker placed in the upper tier of the
distribution, while routing `uncertain` and `likely_hallucinated` to claims where
retrieval could not surface high-quality evidence. The original TAU = 0.5 from initial
development was operationally inert on this corpus — every retrieved passage exceeded it —
and produced false `knowledge_gap` verdicts on out-of-KB claims that should have been
flagged as `likely_hallucinated`.

TAU is still marked `TUNE_ME`. Day 4 will refine it with a proper dev-set sweep over
`dev_final_cleaned_testset.jsonl`, jointly with retrieval recall metrics. The current
value is a defensible first calibration based on observed score distributions, not a
final production threshold.

**REFUTES and TAU.** The REFUTES branch does not consult TAU: refutation is a direct NLI
prediction and already implies topical relevance — the passage had to be on-topic for the
model to confidently predict REFUTES.

**Return.** `{claim, verdict, nli_label, nli_conf, similarity, evidence_passage,
evidence_page_id, all_passages}` where `all_passages` retains the full

In [18]:
# Cell 6E -- final_verdict: end-to-end claim verdict
# Depends on: score_passages (Step 6C), aggregate_passages (Step 6D)

TAU       = 0.85    # TUNE_ME: similarity threshold on sigmoid scale [0,1]
                   # 0.85 ;tuned in Day 4
NEI_LABEL = "NOT ENOUGH INFO"   # full string -- must match NLI_ID2LABEL output exactly


def final_verdict(claim: str) -> dict:
    """
    Full pipeline: retrieve -> NLI -> aggregate -> 5-case verdict.

    Returns
    -------
    dict:
        claim            : str
        verdict          : "factual" | "hallucinated" | "uncertain"
                           | "knowledge_gap" | "likely_hallucinated"
        nli_label        : "SUPPORTS" | "REFUTES" | "NOT ENOUGH INFO"
        nli_conf         : float
        similarity       : float  (sigmoid of reranker score, in [0, 1])
        evidence_passage : str
        evidence_page_id : str
        all_passages     : list   (full per-passage scored list from Step 6C)
    """
    scored = score_passages(claim)
    best   = aggregate_passages(scored)
    label  = best["predicted_label"]
    conf   = best["predicted_conf"]
    sim    = best["similarity"]

    if label == "REFUTES":
        verdict = "hallucinated"
    elif label == "SUPPORTS" and sim >= TAU:
        verdict = "factual"
    elif label == "SUPPORTS" and sim < TAU:
        verdict = "uncertain"
    elif label == NEI_LABEL and sim >= TAU:
        verdict = "knowledge_gap"
    else:  # NEI_LABEL and sim < TAU
        verdict = "likely_hallucinated"

    return {
        "claim":            claim,
        "verdict":          verdict,
        "nli_label":        label,
        "nli_conf":         round(conf, 4),
        "similarity":       round(sim, 4),
        "evidence_passage": best["passage_text"],
        "evidence_page_id": best["page_id"],
        "all_passages":     scored,
    }


# ── Quick sanity test ────────────────────────────────────────────────────────
_fv_claim = "Albert Einstein was awarded the Nobel Prize in Physics."
_fv       = final_verdict(_fv_claim)
print("final_verdict sanity:\n")
print(f"  Claim   : {_fv['claim']}")
print(f"  Verdict : {_fv['verdict']}")
print(f"  NLI     : {_fv['nli_label']}  conf={_fv['nli_conf']:.2%}")
print(f"  Sim     : {_fv['similarity']:.4f}  (TAU={TAU})")
print(f"  Page    : {_fv['evidence_page_id'][:70]}")
print(f"\nAll passages after reranking:")
for _i, _p in enumerate(_fv["all_passages"], 1):
    print(f"  {_i}. {_p['predicted_label']:15s} "
          f"conf={_p['predicted_conf']:.2%}  "
          f"sim={_p['similarity']:.3f}  "
          f"{_p['page_id'][:60]}")
print(f"\nStep 6E: PASS")

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

final_verdict sanity:

  Claim   : Albert Einstein was awarded the Nobel Prize in Physics.
  Verdict : factual
  NLI     : SUPPORTS  conf=91.62%
  Sim     : 0.9531  (TAU=0.85)
  Page    : Einstein's_awards_and_honors

All passages after reranking:
  1. SUPPORTS        conf=91.62%  sim=0.953  Einstein's_awards_and_honors
  2. NOT ENOUGH INFO conf=96.45%  sim=0.888  List_of_Jewish_American_physicists
  3. NOT ENOUGH INFO conf=66.33%  sim=0.882  Percy_W._Bridgman_House
  4. NOT ENOUGH INFO conf=88.32%  sim=0.881  Dalton_Medal
  5. NOT ENOUGH INFO conf=98.11%  sim=0.876  Reform_Congregation_Keneseth_Israel_-LRB-Philadelphia-RRB-

Step 6E: PASS


In [19]:
_claim = "Albert Einstein was awarded the Nobel Prize in Physics."

print("BM25 top-10:")
for pid, score in bm25_search(_claim, k=10):
    print(f"  {score:.2f}  {pid}")

print("\nDense top-10:")
for pid, score in dense_search(_claim, k=10):
    print(f"  {score:.4f}  {pid}")

BM25 top-10:


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  16.65  Einstein_Prize::7
  15.44  Albert_Einstein_Peace_Prize::0
  15.07  List_of_Jewish_American_physicists::22
  14.69  Einstein's_awards_and_honors::0
  14.31  Dalton_Medal::31
  14.31  2011_Nobel_Prize_in_Physics::0
  14.31  Alexei_Alexeyevich_Abrikosov::1
  14.31  Dalton_Medal::40
  14.10  Curie_family::1
  13.82  Gravitational-wave_observatory::9

Dense top-10:
  0.6278  Einstein's_awards_and_honors::0
  0.5883  Albert_Einstein_World_Award_of_Science::1
  0.5844  Photoelectric_effect::25
  0.5837  Albert_Einstein_Award::0
  0.5832  Percy_W._Bridgman_House::14
  0.5770  ETH_Zurich::9
  0.5738  Wolfgang_Pauli::1
  0.5684  Invar::5
  0.5678  Reform_Congregation_Keneseth_Israel_-LRB-Philadelphia-RRB-::19
  0.5668  Brownian_motion::7


### Step 6F - Smoke test — 5 claims with per-stage timing

Five hand-picked claims designed to exercise each of the five verdict branches. The loop
inlines `retrieve_top5` → `nli_predict_batch` → `aggregate_passages` with `time.time()`
deltas so each stage can be timed independently.

| Claim | Expected verdict |
|---|---|
| Albert Einstein was awarded the Nobel Prize in Physics in 1921. | `factual` |
| The Eiffel Tower is located in Berlin, Germany. | `hallucinated` |
| Tetris was created by Alexey Pajitnov in the Soviet Union. | `factual` or `uncertain` |
| Anthropic was founded in 2021. | `likely_hallucinated` (post-FEVER wiki cutoff) |
| Water boils at 100 degrees Celsius at sea level. | `factual` or `knowledge_gap` |

A summary line at the end reports average per-claim retrieve / NLI / total times and
smoke test wall-clock total. The retrieve stage (BM25 O(25M) + dense query) is the
expected bottleneck; the NLI stage should be 3–5× faster than the pre-batch baseline.

In [20]:
# Cell 6F -- Smoke test: 5 claims with per-stage timing
# Inlines retrieve + NLI + aggregate to isolate stage latencies.
# Depends on: retrieve_top5 (5F), nli_predict_batch (6B), aggregate_passages (6D), TAU (6E)
import time, math
from collections import Counter as _Ctr

SMOKE_CLAIMS = [
    ("Albert Einstein was awarded the Nobel Prize in Physics in 1921.",
     "target: factual"),
    ("The Eiffel Tower is located in Berlin, Germany.",
     "target: hallucinated"),
    ("Tetris was created by Alexey Pajitnov in the Soviet Union.",
     "target: factual or uncertain (obscure source)"),
    ("Anthropic was founded in 2021.",
     "target: likely_hallucinated  (post-FEVER wiki cutoff)"),
    ("Water boils at 100 degrees Celsius at sea level.",
     "target: factual or knowledge_gap  (general science fact)"),
]

NEI_LABEL      = "NOT ENOUGH INFO"
SEP            = "=" * 72
_t_smoke_start = time.time()
_timings       = []   # [(t_retrieve, t_nli, t_total), ...]

print(SEP)
print("SMOKE TEST  --  5 claims with per-stage timing")
print(SEP)

for _smoke_claim, _smoke_comment in SMOKE_CLAIMS:
    print(f"\n[{_smoke_comment}]")

    # Stage 1: retrieval
    _t0        = time.time()
    _retrieved = retrieve_top5(_smoke_claim)
    _t_ret     = time.time() - _t0

    # Stage 2: batched NLI
    _t1        = time.time()
    _texts     = [_text for _, _text, _ in _retrieved]
    _nli_batch = nli_predict_batch(_smoke_claim, _texts)
    _scored    = []
    for (_pid, _text, _rs), _nli in zip(_retrieved, _nli_batch):
        _sim   = 1.0 / (1.0 + math.exp(-_rs))
        _parts = _pid.rsplit("::", 1)
        _scored.append({
            "passage_text":    _text,
            "page_id":         _parts[0] if len(_parts) == 2 else _pid,
            "sentence_id":     _parts[1] if len(_parts) == 2 else "?",
            "rerank_score":    round(_rs, 4),
            "similarity":      round(_sim, 4),
            "probs":           _nli["probs"],
            "predicted_label": _nli["label"],
            "predicted_conf":  _nli["confidence"],
        })
    _t_nli = time.time() - _t1

    # Stage 3: aggregate + verdict (negligible)
    _best = aggregate_passages(_scored)
    _lbl  = _best["predicted_label"]
    _sim  = _best["similarity"]
    if _lbl == "REFUTES":
        _verdict = "hallucinated"
    elif _lbl == "SUPPORTS" and _sim >= TAU:
        _verdict = "factual"
    elif _lbl == "SUPPORTS" and _sim < TAU:
        _verdict = "uncertain"
    elif _lbl == NEI_LABEL and _sim >= TAU:
        _verdict = "knowledge_gap"
    else:
        _verdict = "likely_hallucinated"
    _t_total = time.time() - _t0
    _timings.append((_t_ret, _t_nli, _t_total))

    # Print
    _vc = _Ctr(p["predicted_label"] for p in _scored)
    print(f"  Claim   : {_smoke_claim}")
    print(f"  Verdict : {_verdict}")
    print(f"  NLI     : {_lbl}  conf={_best['predicted_conf']:.2%}")
    print(f"  Sim     : {_sim:.4f}  (TAU={TAU})")
    print(f"  Page    : {_best['page_id'][:68]}")
    print(f"  Passage : {_best['passage_text'][:100]}")
    print(f"  Timing  : retrieve={_t_ret:.1f}s  nli={_t_nli:.1f}s  total={_t_total:.1f}s")
    print(f"  Votes   : {dict(_vc)}")
    for _si, _sp in enumerate(_scored, 1):
        print(
            f"    {_si}. {_sp['predicted_label']:15s} "
            f"conf={_sp['predicted_conf']:.2%}  "
            f"sim={_sp['similarity']:.3f}  "
            f"{_sp['page_id'][:48]}"
        )

# Summary
_n        = len(_timings)
_avg_ret  = sum(t[0] for t in _timings) / _n
_avg_nli  = sum(t[1] for t in _timings) / _n
_avg_tot  = sum(t[2] for t in _timings) / _n
_smoke_t  = time.time() - _t_smoke_start

print()
print(SEP)
print(f"Avg per claim:  retrieve={_avg_ret:.1f}s  nli={_avg_nli:.1f}s  total={_avg_tot:.1f}s")
print(f"Smoke test total: {_smoke_t:.1f}s")
print("Step 6F: SMOKE TEST COMPLETE")
print(SEP)
print("\nAll passages after reranking:")
for i, p in enumerate(_fv["all_passages"], 1):
    print(f"  {i}. {p['predicted_label']:15s} sim={p['similarity']:.3f}  {p['page_id'][:60]}")

SMOKE TEST  --  5 claims with per-stage timing

[target: factual]


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Claim   : Albert Einstein was awarded the Nobel Prize in Physics in 1921.
  Verdict : factual
  NLI     : SUPPORTS  conf=83.70%
  Sim     : 0.9700  (TAU=0.85)
  Page    : Einstein's_awards_and_honors
  Passage : In 1922 , Albert Einstein was awarded the 1921 Nobel Prize in Physics , `` for his services to Theor
  Timing  : retrieve=6.8s  nli=0.2s  total=7.0s
  Votes   : {'SUPPORTS': 3, 'NOT ENOUGH INFO': 2}
    1. SUPPORTS        conf=83.70%  sim=0.970  Einstein's_awards_and_honors
    2. NOT ENOUGH INFO conf=94.27%  sim=0.930  List_of_Jewish_American_physicists
    3. SUPPORTS        conf=91.33%  sim=0.909  Photoelectric_effect
    4. SUPPORTS        conf=92.43%  sim=0.906  Albert_Einstein
    5. NOT ENOUGH INFO conf=97.90%  sim=0.898  Matteucci_Medal

[target: hallucinated]


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Claim   : Anthropic was founded in 2021.
  Verdict : hallucinated
  NLI     : REFUTES  conf=96.84%
  Sim     : 0.8429  (TAU=0.85)
  Page    : The_Climate_Mobilization
  Passage : It was founded by psychologist Margaret Klein Salamon to confront climate change denial and build th
  Timing  : retrieve=26.8s  nli=0.1s  total=26.9s
  Votes   : {'REFUTES': 2, 'NOT ENOUGH INFO': 3}
    1. REFUTES         conf=96.84%  sim=0.843  The_Climate_Mobilization
    2. NOT ENOUGH INFO conf=98.21%  sim=0.808  Bryant_Township,_Logan_County,_North_Dakota
    3. NOT ENOUGH INFO conf=95.72%  sim=0.786  Agricultural_Society_of_Baton_Rouge
    4. NOT ENOUGH INFO conf=98.04%  sim=0.769  Robin_and_the_7_Hoods_-LRB-album-RRB-
    5. REFUTES         conf=95.94%  sim=0.768  Anthropic_principle

[target: factual or knowledge_gap  (general science fact)]


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Claim   : Water boils at 100 degrees Celsius at sea level.
  Verdict : factual
  NLI     : SUPPORTS  conf=92.30%
  Sim     : 0.9426  (TAU=0.85)
  Page    : Boiling_point
  Passage : For example , water boils at 100 C at sea level , but at 93.4 C at 2000 m altitude .
  Timing  : retrieve=20.8s  nli=0.1s  total=20.9s
  Votes   : {'SUPPORTS': 2, 'REFUTES': 1, 'NOT ENOUGH INFO': 2}
    1. SUPPORTS        conf=92.30%  sim=0.943  Boiling_point
    2. REFUTES         conf=69.23%  sim=0.903  Steam
    3. SUPPORTS        conf=75.45%  sim=0.900  Boiling
    4. NOT ENOUGH INFO conf=87.77%  sim=0.872  Hyperthermophile
    5. NOT ENOUGH INFO conf=55.54%  sim=0.854  Psychrometric_constant

Avg per claim:  retrieve=20.6s  nli=0.1s  total=20.7s
Smoke test total: 103.7s
Step 6F: SMOKE TEST COMPLETE

All passages after reranking:
  1. SUPPORTS        sim=0.953  Einstein's_awards_and_honors
  2. NOT ENOUGH INFO sim=0.888  List_of_Jewish_American_physicists
  3. NOT ENOUGH INFO sim=0.882  Percy_W._Bridg